# Payment Infrastructure, Financial Sanctions, and Cross-Border Currency Diversification
## Panel Evidence from 170 Countries, 2010–2023

**Panel:** 170 countries × 2010–2023 (N = 2,380 country-year observations)  
**Methods:** TWFE · Event Study · CS-DiD (Callaway–Sant'Anna 2021) · PPML · IV · Push×Pull Interaction  

### Key Findings
- CIPS accession significantly increases offshore RMB clearing-bank adoption  
  (CS-DiD preferred estimate: **+5.2 pp**, p ≈ 0.04)  
- Push×Pull interaction: **α₃ = +1.77** (p = 0.006) in core TWFE;  
  amplifies **7–13×** post-2022 (Phase 2: α₃ = +20.8, p = 0.018)  
- Sanctions alone **reinforce** dollar usage when RMB infrastructure is absent  
  (α₂ = −1.57, p = 0.015) — the Sanctions Paradox  

> **Data:** Place `master_panel.csv` in the working directory before running.  
> All outputs are saved to `output/`.


## 1. Environment Setup

In [ ]:
# ================================================================
# CELL 0: Environment Setup
# ================================================================
import subprocess, sys, warnings, os
warnings.filterwarnings('ignore')

for pkg in ['linearmodels']:
    try: __import__(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])

import numpy as np
import pandas as pd
from scipy import stats
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import matplotlib.patheffects as pe
import matplotlib.gridspec as gridspec
import statsmodels.api as sm
from linearmodels.panel import PanelOLS

# ---- Standard academic sans-serif style ----
plt.rcParams.update({
    'figure.dpi': 150,
    'savefig.dpi': 300,
    'font.size': 10,
    'font.family': 'sans-serif',
    'font.sans-serif': ['DejaVu Sans', 'Arial', 'Helvetica'],
    'axes.titlesize': 12,
    'axes.labelsize': 11,
    'xtick.labelsize': 9,
    'ytick.labelsize': 9,
    'legend.fontsize': 9,
    'figure.facecolor': 'white',
    'axes.grid': True,
    'grid.alpha': 0.3,
    'grid.linestyle': '--',
})

OUT = 'output'
os.makedirs(OUT, exist_ok=True)
print(f'Environment ready. Output dir: {OUT}')

## 2. Data Loading and Variable Construction

In [ ]:
# ================================================================
# CELL 1: Load Panel & Construct Variables
# ================================================================
PANEL_FILE = 'master_panel.csv'
if not os.path.exists(PANEL_FILE):
    for alt in ['master_panel.csv', 'master_panel.csv']:
        if os.path.exists(alt):
            PANEL_FILE = alt; break

df = pd.read_csv(PANEL_FILE, low_memory=False)
df.replace([np.inf, -np.inf], np.nan, inplace=True)
print(f'Loaded: {PANEL_FILE}, shape = {df.shape}')

SANC = 'sanction_fin_norm_fixed'
CONTROLS = ['ln_gdp','ln_trade_openness','financial_depth_it',
            'capital_openness_it_filled','gdp_growth_it','inflation_it_w',
            'fdi_inflow_gdp_it_w','unga_align_norm_it']
CS = ' + '.join(CONTROLS)

# Treatment timing
first_treat = df.loc[df['CIPSit']==1].groupby('iso3')['year'].min()
df['g_first_treat'] = df['iso3'].map(first_treat)
df['treated_direct'] = df['g_first_treat'].notna().astype(int)
df['trend_direct'] = df['treated_direct'] * (df['year'] - df['year'].min())

# Interaction terms
df['infra_x_sanction'] = df['rmb_infra_it'].fillna(0) * df[SANC].fillna(0)
df['cips_x_sanction']  = df['CIPSit'].fillna(0) * df[SANC].fillna(0)
df['clear_x_sanction'] = df['clearing_bank_it'].fillna(0) * df[SANC].fillna(0)
df['post2022'] = (df['year'] >= 2022).astype(int)

# DV transforms
if 'rmb_invoicing_share' in df.columns:
    df['rmb_invoicing_share'] = pd.to_numeric(df['rmb_invoicing_share'], errors='coerce')
    df.loc[df['rmb_invoicing_share'] < 0, 'rmb_invoicing_share'] = np.nan
    hi = df['rmb_invoicing_share'].quantile(0.995)
    df['rmb_invoicing_w'] = df['rmb_invoicing_share'].clip(upper=hi)
    df['rmb_inv_asinh'] = np.arcsinh(df['rmb_invoicing_share'])

# Role classification
SENDERS = ['USA','GBR','FRA','DEU','CAN','AUS','JPN']
TARGETS = ['RUS','IRN','PRK','SYR','CUB','VEN','MMR','BLR']
df['role'] = 'Observer'
df.loc[df['iso3'].isin(SENDERS), 'role'] = 'Sender'
df.loc[df['iso3'].isin(TARGETS), 'role'] = 'Target'

# Region mapping
REGION_MAP = {}
_af=['DZA','AGO','BEN','BWA','BFA','BDI','CPV','CMR','CAF','TCD','COM','COD',
     'COG','CIV','DJI','EGY','GNQ','ERI','SWZ','ETH','GAB','GMB','GHA','GIN',
     'GNB','KEN','LSO','LBR','LBY','MDG','MWI','MLI','MRT','MUS','MAR','MOZ',
     'NAM','NER','NGA','RWA','STP','SEN','SYC','SLE','SOM','ZAF','SSD','SDN',
     'TZA','TGO','TUN','UGA','ZMB','ZWE']
_as=['AFG','ARM','AZE','BHR','BGD','BTN','BRN','KHM','CHN','GEO','IND','IDN',
     'IRN','IRQ','ISR','JPN','JOR','KAZ','KWT','KGZ','LAO','LBN','MYS','MDV',
     'MNG','MMR','NPL','OMN','PAK','PHL','QAT','SAU','SGP','KOR','LKA','SYR',
     'TJK','THA','TLS','TUR','TKM','ARE','UZB','VNM','YEM','TWN','HKG','MAC']
_eu=['ALB','AND','AUT','BLR','BEL','BIH','BGR','HRV','CYP','CZE','DNK','EST',
     'FIN','FRA','DEU','GRC','HUN','ISL','IRL','ITA','LVA','LTU','LUX','MLT',
     'MDA','MNE','NLD','MKD','NOR','POL','PRT','ROU','RUS','SRB','SVK','SVN',
     'ESP','SWE','CHE','UKR','GBR']
_am=['ARG','BHS','BRB','BLZ','BOL','BRA','CAN','CHL','COL','CRI','CUB','DOM',
     'ECU','SLV','GTM','GUY','HTI','HND','JAM','MEX','NIC','PAN','PRY','PER',
     'SUR','TTO','URY','VEN','USA']
_oc=['AUS','FJI','NZL','PNG','WSM','SLB','TON','VUT']
for c in _af: REGION_MAP[c]='Africa'
for c in _as: REGION_MAP[c]='Asia'
for c in _eu: REGION_MAP[c]='Europe'
for c in _am: REGION_MAP[c]='Americas'
for c in _oc: REGION_MAP[c]='Oceania'
df['region'] = df['iso3'].map(REGION_MAP).fillna('Other')

# Regional CIPS instrument
if 'regional_cips_share' not in df.columns:
    rcs = []
    for _, row in df.iterrows():
        reg = REGION_MAP.get(row['iso3'], 'Other')
        same = df[(df['region']==reg)&(df['year']==row['year'])&(df['iso3']!=row['iso3'])]
        rcs.append(same['CIPSit'].mean() if len(same)>0 else 0)
    df['regional_cips_share'] = rcs

print(f'Panel: {df.shape[0]} obs, {df["iso3"].nunique()} countries, '
      f'{df["year"].min()}-{df["year"].max()}')

### 2.1 CIPS Cohort Composition Verification

In [ ]:
# ================================================================
# CIPS COHORT VERIFICATION
# ================================================================
# Identify the first year each country has CIPSit == 1
cips_countries = df[df['CIPSit'] == 1].groupby('iso3')['year'].min().reset_index()
cips_countries.columns = ['iso3', 'first_cips_year']

print('='*70)
print('CIPS COHORT COMPOSITION')
print('='*70)

for yr in sorted(cips_countries['first_cips_year'].unique()):
    cohort = cips_countries[cips_countries['first_cips_year'] == yr]
    countries = sorted(cohort['iso3'].tolist())
    print(f'\nCohort {yr} (N={len(countries)}):')
    print(f'  {", ".join(countries)}')

    # Cross-check with country names if available
    if 'country_name' in df.columns:
        names = df[df['iso3'].isin(countries)][['iso3','country_name']].drop_duplicates()
        for _, row in names.sort_values('iso3').iterrows():
            print(f'    {row["iso3"]}: {row["country_name"]}')

# Also check: are there countries that NEVER have CIPSit==1?
never_cips = set(df['iso3'].unique()) - set(cips_countries['iso3'])
print(f'\nNever-treated countries: {len(never_cips)}')

# Verify totals
print(f'\nTotal treated: {len(cips_countries)}')
print(f'Cohort sizes: {dict(cips_countries["first_cips_year"].value_counts().sort_index())}')
print(f'Paper claims: 2015:8, 2016:32, 2018:1')

### 2.2 BRICS+ Expansion Member Profiles (2021–2023)

In [ ]:
# ================================================================
# TABLE 9.1 VERIFICATION: BRICS+ Expansion Members
# ================================================================
expansion = {
    'EGY': 'Egypt',
    'ETH': 'Ethiopia',
    'IRN': 'Iran',
    'SAU': 'Saudi Arabia',
    'ARE': 'UAE',
    'IDN': 'Indonesia'
}

cols_show = ['rmb_invoicing_w', 'clearing_bank_it', 'swap_line_it',
             'CIPSit', SANC, 'china_trade_share_it']

# Observer mean for comparison
obs = df[(df['role']=='Observer') & (df['year'].between(2021,2023))]

print('='*90)
print('TABLE 9.1 VERIFICATION: Pre-Accession Profiles (2021-2023 Means)')
print('='*90)
print(f'{"Country":<16}', end='')
for c in cols_show:
    label = c.replace('_it','').replace('rmb_invoicing_w','RMB_Inv')
    label = label.replace('clearing_bank','ClearBank').replace('swap_line','SwapLine')
    label = label.replace('CIPSit','CIPS').replace(SANC,'Sanctions')
    label = label.replace('china_trade_share','ChinaTrade')
    print(f'{label:>12}', end='')
print()
print('-'*90)

for iso, name in expansion.items():
    sub = df[(df['iso3']==iso) & (df['year'].between(2021,2023))]
    print(f'{name:<16}', end='')
    for c in cols_show:
        if c in sub.columns and sub[c].notna().any():
            val = sub[c].mean()
            print(f'{val:>12.3f}', end='')
        else:
            print(f'{"—":>12}', end='')
    print(f'  (N={len(sub)})')

# Observer mean
print('-'*90)
print(f'{"Observer mean":<16}', end='')
for c in cols_show:
    if c in obs.columns and obs[c].notna().any():
        val = obs[c].mean()
        print(f'{val:>12.3f}', end='')
    else:
        print(f'{"—":>12}', end='')
print(f'  (N_countries={obs["iso3"].nunique()})')

# Also check: is 'china_trade_share_it' available for SAU?
print('\n--- china_trade_share_it availability check ---')
for iso, name in expansion.items():
    sub = df[(df['iso3']==iso) & (df['year'].between(2021,2023))]
    if 'china_trade_share_it' in sub.columns:
        n_valid = sub['china_trade_share_it'].notna().sum()
        print(f'  {name}: {n_valid}/{len(sub)} non-missing values')
    else:
        print(f'  {name}: column not found')

## 3. Helper Functions

In [ ]:
# ================================================================
# CELL 2: Helper Functions
# ================================================================
def fit_twfe(data, formula):
    dp = data.set_index(['iso3','year']).sort_index()
    return PanelOLS.from_formula(formula, dp, drop_absorbed=True).fit(
        cov_type='clustered', cluster_entity=True)

def extract_row(model, var, label, spec, dv=''):
    if model is None or var not in model.params.index: return None
    b, se, p = model.params[var], model.std_errors[var], model.pvalues[var]
    stars = '***' if p<0.01 else '**' if p<0.05 else '*' if p<0.10 else ''
    return {'DV':dv, 'spec':spec, 'label':label, 'var':var,
            'beta':round(b,4), 'se':round(se,4), 'p':round(p,4),
            'stars':stars, 'n':int(model.nobs), 'r2':round(float(model.rsquared),4)}

def save_table(rows, name):
    out = pd.DataFrame([r for r in rows if r is not None])
    path = os.path.join(OUT, name+'.csv')
    out.to_csv(path, index=False)
    print(f'  Saved: {path}  ({len(out)} rows)')
    return out

print('Helper functions defined.')

## 4. Core Regression Tables

### Table 1: CIPS Accession → RMB Infrastructure (TWFE)
Estimates the effect of CIPS membership on clearing-bank adoption and swap-line presence,
with and without group-specific linear trends.

In [ ]:
# ================================================================
# CELL 3: TABLE 1 — CIPS → Infrastructure
# ================================================================
print('='*70); print('TABLE 1: CIPS → RMB Infrastructure (TWFE)'); print('='*70)
rows_t1 = []
for dv, label in [('clearing_bank_it','Clearing bank (0/1)'),
                   ('swap_line_it','Swap line (0/1)'),
                   ('rmb_infra_it','RMB infra index (0-2)')]:
    if dv not in df.columns: continue
    fa = dv+' ~ CIPSit + '+CS+' + EntityEffects + TimeEffects'
    ma = fit_twfe(df, fa)
    rows_t1.append(extract_row(ma,'CIPSit',label,'(A) Baseline',dv))
    fb = dv+' ~ CIPSit + trend_direct + '+CS+' + EntityEffects + TimeEffects'
    mb = fit_twfe(df, fb)
    rows_t1.append(extract_row(mb,'CIPSit',label,'(B) + Group trend',dv))
    rows_t1.append(extract_row(mb,'trend_direct',label+' [trend]','(B) + Group trend',dv))
table1 = save_table(rows_t1, 'table1_core_cips_infrastructure')
display(table1)

### Table 2: Push × Pull Interaction — RMB Invoicing Share
Core specification testing whether sanctions exposure translates into RMB invoicing
diversification conditional on RMB payment infrastructure.

In [ ]:
# ================================================================
# CELL 4: TABLE 2 — Push × Pull Interaction (Central Result)
# ================================================================
print('='*70); print('TABLE 2: Push × Pull → RMB Invoicing'); print('='*70)
rows_t2 = []
for dv, dlab in [('rmb_invoicing_w','wins.'),('rmb_inv_asinh','asinh')]:
    if dv not in df.columns: continue
    # (A) Infra × Sanctions
    fa = dv+' ~ rmb_infra_it + '+SANC+' + infra_x_sanction + '+CS+' + EntityEffects + TimeEffects'
    ma = fit_twfe(df, fa)
    rows_t2.append(extract_row(ma,'rmb_infra_it','Pull: RMB infra',f'(A) Infra×Sanc [{dlab}]',dv))
    rows_t2.append(extract_row(ma,SANC,'Push: Sanctions',f'(A) Infra×Sanc [{dlab}]',dv))
    rows_t2.append(extract_row(ma,'infra_x_sanction','Push×Pull',f'(A) Infra×Sanc [{dlab}]',dv))
    # (B) CIPS × Sanctions
    fb = dv+' ~ CIPSit + '+SANC+' + cips_x_sanction + trend_direct + '+CS+' + EntityEffects + TimeEffects'
    mb = fit_twfe(df, fb)
    rows_t2.append(extract_row(mb,'CIPSit','Pull: CIPS',f'(B) CIPS×Sanc [{dlab}]',dv))
    rows_t2.append(extract_row(mb,SANC,'Push: Sanctions',f'(B) CIPS×Sanc [{dlab}]',dv))
    rows_t2.append(extract_row(mb,'cips_x_sanction','CIPS×Sanctions',f'(B) CIPS×Sanc [{dlab}]',dv))
    # (C) Clearing × Sanctions
    fc = dv+' ~ clearing_bank_it + '+SANC+' + clear_x_sanction + '+CS+' + EntityEffects + TimeEffects'
    mc = fit_twfe(df, fc)
    rows_t2.append(extract_row(mc,'clearing_bank_it','Pull: Clearing bank',f'(C) Clear×Sanc [{dlab}]',dv))
    rows_t2.append(extract_row(mc,SANC,'Push: Sanctions',f'(C) Clear×Sanc [{dlab}]',dv))
    rows_t2.append(extract_row(mc,'clear_x_sanction','Clearing×Sanctions',f'(C) Clear×Sanc [{dlab}]',dv))
table2 = save_table(rows_t2, 'table2_push_pull_interaction')
display(table2)

## 5. Event Study
Dynamic treatment effects around CIPS accession across the 2015, 2016, and 2018 cohorts.
Pre-treatment coefficients test the parallel trends assumption.

In [ ]:
# ================================================================
# CELL 5: Event Study
# ================================================================
print('='*70); print('FIGURE 1: Event Study'); print('='*70)
df_es = df.copy()
et = np.where(df_es['g_first_treat'].notna(),
              df_es['year'] - df_es['g_first_treat'], np.nan)
df_es['event_time_c'] = pd.Series(et, index=df_es.index).clip(-5, 7)

etvals = sorted(int(v) for v in df_es['event_time_c'].dropna().unique() if v != -1)
evtcols = []
for v in etvals:
    col = 'Em'+str(abs(v)) if v<0 else ('E0' if v==0 else 'Ep'+str(v))
    df_es[col] = (df_es['event_time_c']==v).astype(float)
    df_es.loc[df_es['event_time_c'].isna(), col] = 0.0
    evtcols.append(col)

dp_es = df_es.set_index(['iso3','year'])
res_es = PanelOLS.from_formula(
    'clearing_bank_it ~ '+' + '.join(evtcols)+' + '+CS+' + EntityEffects + TimeEffects',
    data=dp_es, drop_absorbed=True
).fit(cov_type='clustered', cluster_entity=True)

cd = res_es.params[evtcols].to_frame('coef')
ci = res_es.conf_int().loc[evtcols]; ci.columns = ['lo','hi']
es_df = cd.join(ci)
def _pt(n):
    if n.startswith('Em'): return -int(n[2:])
    if n.startswith('Ep'): return int(n[2:])
    return 0
es_df['t'] = es_df.index.map(_pt)
es_df = es_df.sort_values('t')

pre = es_df[es_df['t']<=-2]
slope, intercept = np.polyfit(pre['t'], pre['coef'], 1)
actual_t0 = es_df.loc[es_df['t']==0, 'coef'].iloc[0]
excess = actual_t0 - intercept
print(f'Pre-trend slope: {slope:.4f}, Extrapolated t=0: {intercept:.4f}')
print(f'Actual t=0: {actual_t0:.4f}, Level shift: {excess:.4f} pp')

fig, ax = plt.subplots(figsize=(10, 5.5))
ax.errorbar(es_df['t'], es_df['coef'],
            yerr=[es_df['coef']-es_df['lo'], es_df['hi']-es_df['coef']],
            fmt='o-', capsize=4, color='steelblue', lw=2, ms=6, zorder=5)
ax.fill_between(es_df['t'], es_df['lo'], es_df['hi'], alpha=0.12, color='steelblue')
xt = np.linspace(-5, 2, 50)
ax.plot(xt, slope*xt+intercept, '--', color='gray', lw=1.5,
        label=f'Pre-trend extrapolation (slope={slope:.3f})')

# Annotation — positioned to avoid overlap
ax.annotate(f'Level shift = {excess:.2f} pp',
            xy=(0, actual_t0), xytext=(3.0, actual_t0+0.12),
            fontsize=9, ha='center',
            arrowprops=dict(arrowstyle='->', color='red', lw=1.5),
            bbox=dict(boxstyle='round,pad=0.3', facecolor='lightyellow'))
ax.axhline(0, color='black', lw=0.8)
ax.axvline(-0.5, color='red', ls='--', alpha=0.6, label='CIPS accession')
ax.set_xlabel('Event time (0 = CIPS accession year)')
ax.set_ylabel('ATT relative to t = −1 (pp)')
ax.set_title('Event Study: Clearing Bank Adoption — Trend vs Discrete Shift Decomposition')
ax.legend(loc='upper left', fontsize=9)
fig.tight_layout()
fig.savefig(os.path.join(OUT,'fig_event_study.png'), dpi=300, bbox_inches='tight')
plt.show()
print('  Saved: fig_event_study.png')

## 6. Callaway–Sant'Anna (2021) Heterogeneity-Robust DiD
Group-time average treatment effects (ATT(g,t)) correcting for treatment-timing
heterogeneity bias in staggered adoption settings. Preferred causal estimator.

In [ ]:
# ================================================================
# CELL 6: CS-DiD
# ================================================================
print('='*70); print('CS-DiD: Callaway-Sant\'Anna (2021)'); print('='*70)
try:
    from csdid.att_gt import ATTgt
    cs_df = df[['iso3','year','clearing_bank_it','g_first_treat']].copy()
    cs_df['g'] = cs_df['g_first_treat'].fillna(0).astype(int)
    id_map = {iso:i for i,iso in enumerate(cs_df['iso3'].unique())}
    cs_df['id'] = cs_df['iso3'].map(id_map)
    cs_df = cs_df.dropna(subset=['clearing_bank_it'])
    model = ATTgt(data=cs_df, yname='clearing_bank_it', tname='year',
                  idname='id', gname='g')
    result = model.fit()
    r = result.results
    att_gt_df = pd.DataFrame({'group':r['group'],'year':r['year'],
                              'att':r['att'],'se':r['se'],'post':r['post ']})
    att_gt_df['event_time'] = att_gt_df['year'] - att_gt_df['group']
    group_sizes = cs_df[cs_df['g']>0].groupby('g')['id'].nunique()
    att_gt_df['n_g'] = att_gt_df['group'].map(group_sizes)
    dyn = att_gt_df.dropna(subset=['se']).groupby('event_time').apply(
        lambda x: pd.Series({
            'att': np.average(x['att'], weights=x['n_g']),
            'se': np.sqrt(np.average(x['se']**2, weights=x['n_g']**2) /
                          np.sum(x['n_g'])**2 * np.sum(x['n_g']**2)),
            'n_groups': len(x)
        })
    ).reset_index()
    dyn['ci_lo'] = dyn['att'] - 1.96*dyn['se']
    dyn['ci_hi'] = dyn['att'] + 1.96*dyn['se']
    # Pre-trend test
    print('\nFormal pre-trend test (t <= -3):')
    for _, row in dyn[dyn['event_time']<=-3].iterrows():
        sig = 'PASS' if row['ci_lo']<=0<=row['ci_hi'] else 'FAIL'
        print(f'  t={int(row["event_time"])}: ATT={row["att"]:.4f} CI=[{row["ci_lo"]:.4f},{row["ci_hi"]:.4f}] {sig}')
    print('Anticipation effects (t=-2,-1):')
    for _, row in dyn[dyn['event_time'].isin([-2,-1])].iterrows():
        print(f'  t={int(row["event_time"])}: ATT={row["att"]:.4f} CI=[{row["ci_lo"]:.4f},{row["ci_hi"]:.4f}]')
    # Plot
    fig, ax = plt.subplots(figsize=(10,5))
    ax.errorbar(dyn['event_time'], dyn['att'], yerr=1.96*dyn['se'],
                fmt='s-', capsize=4, color='darkgreen', lw=2, ms=6, label='CS-DiD ATT(e)')
    ax.axvspan(-2.5, -0.5, alpha=0.08, color='orange', label='Anticipation window')
    ax.axhline(0, color='black', lw=0.8)
    ax.axvline(-0.5, color='red', ls='--', alpha=0.6, label='CIPS accession')
    ax.set_xlabel('Event time'); ax.set_ylabel('ATT')
    ax.set_title('Callaway-Sant\'Anna Dynamic ATT: CIPS → Clearing Bank')
    ax.legend(fontsize=8, loc='upper left')
    fig.tight_layout()
    fig.savefig(os.path.join(OUT,'fig_csdid_dynamic.png'), dpi=300, bbox_inches='tight')
    plt.show()
    dyn.to_csv(os.path.join(OUT,'csdid_dynamic_att.csv'), index=False)
except Exception as e:
    print(f'CS-DiD skipped: {e}')

## 7. Poisson Pseudo-Maximum Likelihood (PPML)
Robustness check for share/count DVs using PPML to handle zero-inflation
and the proportional nature of invoicing shares.

In [ ]:
# ================================================================
# CELL 7: PPML
# ================================================================
print('='*70); print('PPML: Push × Pull → RMB Invoicing'); print('='*70)
ppml_cols = ['rmb_invoicing_share','rmb_infra_it','CIPSit',SANC,
             'infra_x_sanction','cips_x_sanction','trend_direct','iso3','year']+CONTROLS
ppml_df = df[[c for c in ppml_cols if c in df.columns]].dropna().copy()
ppml_df['y'] = ppml_df['rmb_invoicing_share'].clip(lower=0).astype(float)
for col in ppml_df.columns:
    if col!='iso3': ppml_df[col] = pd.to_numeric(ppml_df[col], errors='coerce')
ppml_df = ppml_df.dropna()
cdums = pd.get_dummies(ppml_df['iso3'], prefix='c', drop_first=True, dtype=float)
ydums = pd.get_dummies(ppml_df['year'], prefix='y', drop_first=True, dtype=float)
rows_ppml = []
for spec_name, x_vars in [('PPML (A) Infra×Sanc', ['rmb_infra_it',SANC,'infra_x_sanction']+CONTROLS),
                           ('PPML (B) CIPS×Sanc', ['CIPSit',SANC,'cips_x_sanction','trend_direct']+CONTROLS)]:
    X = pd.concat([ppml_df[x_vars].astype(float), cdums, ydums], axis=1)
    X = sm.add_constant(X)
    m = sm.GLM(ppml_df['y'].values, X.values.astype(float),
               family=sm.families.Poisson()).fit(maxiter=300)
    cn = list(X.columns)
    key_vars = x_vars[:3]
    for v in key_vars:
        idx = cn.index(v)
        b, se, p = m.params[idx], m.bse[idx], m.pvalues[idx]
        stars = '***' if p<0.01 else '**' if p<0.05 else '*' if p<0.1 else ''
        rows_ppml.append({'spec':spec_name, 'var':v,
                          'beta':round(b,4), 'se':round(se,4), 'p':round(p,4), 'stars':stars})
        print(f'  {spec_name} | {v}: {b:+.4f} (se={se:.4f}, p={p:.4f}) {stars}')
ppml_table = pd.DataFrame(rows_ppml)
ppml_table.to_csv(os.path.join(OUT,'table_ppml.csv'), index=False)
print('Note: PPML interaction not significant — exponential link absorbs multiplicative effects.')

## 8. Instrumental Variable (IV) Estimation
Instruments individual CIPS accession with the share of regional peers already in CIPS,
addressing potential endogeneity in infrastructure adoption.

In [ ]:
# ================================================================
# CELL 8: IV (Appendix)
# ================================================================
print('='*70); print('IV: Regional CIPS Diffusion (Appendix)'); print('='*70)
iv_cols = ['iso3','year','clearing_bank_it','CIPSit','regional_cips_share','trend_direct']+CONTROLS
iv_df = df[[c for c in iv_cols if c in df.columns]].dropna()
iv_panel = iv_df.set_index(['iso3','year'])
try:
    fs = PanelOLS.from_formula('CIPSit ~ regional_cips_share + trend_direct + '+CS+' + EntityEffects + TimeEffects',
                               iv_panel, drop_absorbed=True).fit(cov_type='clustered', cluster_entity=True)
    fs_beta, fs_se = fs.params['regional_cips_share'], fs.std_errors['regional_cips_share']
    f_stat = (fs_beta/fs_se)**2
    print(f'First stage: beta={fs_beta:.4f}, SE={fs_se:.4f}, F={f_stat:.2f}')
    iv_panel['CIPSit_hat'] = fs.predict().fitted_values
    ss = PanelOLS.from_formula('clearing_bank_it ~ CIPSit_hat + trend_direct + '+CS+' + EntityEffects + TimeEffects',
                               iv_panel, drop_absorbed=True).fit(cov_type='clustered', cluster_entity=True)
    print(f'Second stage: beta={ss.params["CIPSit_hat"]:.4f}, p={ss.pvalues["CIPSit_hat"]:.4f}')
    print(f'WEAK INSTRUMENT (F={f_stat:.1f} < 10). Reported for transparency only.')
    pd.DataFrame([{'first_stage_beta':round(fs_beta,4),'first_stage_se':round(fs_se,4),
                    'first_stage_F':round(f_stat,2),'second_stage_beta':round(ss.params['CIPSit_hat'],4),
                    'second_stage_se':round(ss.std_errors['CIPSit_hat'],4),
                    'second_stage_p':round(ss.pvalues['CIPSit_hat'],4),
                    'N':int(fs.nobs)}]).to_csv(os.path.join(OUT,'table_iv.csv'), index=False)
except Exception as e:
    print(f'IV failed: {e}')

## 9. Heterogeneity Analysis

### Table 3: Subgroup Splits
Sample splits by sanctions exposure history, BRICS membership, and China trade intensity.

In [ ]:
# ================================================================
# CELL 9: TABLE 4 — Sanctions Paradox Temporal
# ================================================================
print('='*70); print('TABLE 4: Sanctions Paradox'); print('='*70)
rows_t4 = []
for pn, mask, lab in [('2010-2021', df['year']<=2021, 'Phase 1'),
                       ('2022-2023', df['year']>=2022, 'Phase 2'),
                       ('Full sample', df['year']>=0, 'Full')]:
    sub = df[mask].copy()
    for dv in ['rmb_invoicing_w','rmb_inv_asinh']:
        if dv not in sub.columns: continue
        try:
            m = fit_twfe(sub, dv+' ~ rmb_infra_it + '+SANC+' + infra_x_sanction + '+CS+' + EntityEffects + TimeEffects')
            spec = pn+' ['+dv+']'
            rows_t4.append(extract_row(m, SANC, lab+': Sanctions', spec, dv))
            rows_t4.append(extract_row(m, 'infra_x_sanction', lab+': Infra×Sanctions', spec, dv))
        except Exception as e: print(f'  {pn} {dv}: {e}')
table4 = save_table(rows_t4, 'table4_sanctions_paradox_temporal')
display(table4)

### Table 4: Sender–Target–Observer Role Taxonomy
Differential effects across the three-role classification:
Senders (G7 sanctioning states), Targets (sanctioned states), and Observer third parties.

In [ ]:
# ================================================================
# CELL 10: TABLE 5 — Heterogeneity
# ================================================================
print('='*70); print('TABLE 5: Heterogeneity'); print('='*70)
rows_t5 = []
if 'ever_sanctioned_it' in df.columns:
    for grp, lab in [(1,'Ever-sanctioned'),(0,'Never-sanctioned')]:
        sub = df[df['ever_sanctioned_it']==grp]
        for dv in ['clearing_bank_it','rmb_invoicing_w']:
            if dv not in sub.columns: continue
            try:
                m = fit_twfe(sub, dv+' ~ CIPSit + trend_direct + '+CS+' + EntityEffects + TimeEffects')
                rows_t5.append(extract_row(m,'CIPSit',lab,f'sanction_split [{dv}]',dv))
            except: pass
if 'brics_member_it' in df.columns:
    for grp, lab in [(1,'BRICS member'),(0,'Non-BRICS')]:
        sub = df[df['brics_member_it']==grp]
        for dv in ['clearing_bank_it','rmb_invoicing_w']:
            if dv not in sub.columns: continue
            try:
                m = fit_twfe(sub, dv+' ~ CIPSit + trend_direct + '+CS+' + EntityEffects + TimeEffects')
                rows_t5.append(extract_row(m,'CIPSit',lab,f'brics_split [{dv}]',dv))
            except: pass
if 'china_trade_share_it' in df.columns:
    ct = df.groupby('iso3')['china_trade_share_it'].mean()
    med = ct.median()
    for isos, lab in [(ct[ct>=med].index,'High China trade'),(ct[ct<med].index,'Low China trade')]:
        sub = df[df['iso3'].isin(isos)]
        for dv in ['clearing_bank_it','rmb_invoicing_w']:
            if dv not in sub.columns: continue
            try:
                m = fit_twfe(sub, dv+' ~ CIPSit + trend_direct + '+CS+' + EntityEffects + TimeEffects')
                rows_t5.append(extract_row(m,'CIPSit',lab,f'trade_split [{dv}]',dv))
            except: pass
table5 = save_table(rows_t5, 'table5_heterogeneity')
display(table5)

# Add to Cell 10:
p1_post_cips = df[(df['year']>=2015) & (df['year']<=2021)].copy()
m_p1_post = fit_twfe(p1_post_cips, formula)
print(f'Phase 1 (2015-2021 only): interaction = {m_p1_post.params["infra_x_sanction"]:.3f}')

## 10. Robustness Checks
Alternative infrastructure measures (CIPS broad definition), sample exclusions
(Russia, China), and DV transformations (arcsinh, minimal controls).

In [ ]:
# ================================================================
# CELL 11: Robustness
# ================================================================
print('='*70); print('TABLE 6: Robustness'); print('='*70)
rows_rob = []
# (A) CIPS broad
if 'CIPS_broad_it' in df.columns:
    for dv in ['clearing_bank_it','rmb_invoicing_w']:
        if dv not in df.columns: continue
        try:
            m = fit_twfe(df, dv+' ~ CIPS_broad_it + '+CS+' + EntityEffects + TimeEffects')
            rows_rob.append(extract_row(m,'CIPS_broad_it',f'CIPS broad [{dv}]','robust_A',dv))
        except: pass
# (B) Drop RUS+CHN
df_nrc = df[~df['iso3'].isin(['RUS','CHN'])]
for dv in ['clearing_bank_it','rmb_invoicing_w']:
    if dv not in df_nrc.columns: continue
    try:
        m = fit_twfe(df_nrc, dv+' ~ CIPSit + trend_direct + '+CS+' + EntityEffects + TimeEffects')
        rows_rob.append(extract_row(m,'CIPSit',f'Drop RUS+CHN [{dv}]','robust_B',dv))
    except: pass
if 'rmb_invoicing_w' in df_nrc.columns:
    try:
        m = fit_twfe(df_nrc, 'rmb_invoicing_w ~ rmb_infra_it + '+SANC+' + infra_x_sanction + '+CS+' + EntityEffects + TimeEffects')
        rows_rob.append(extract_row(m,'infra_x_sanction','Infra×Sanc (no RUS/CHN)','robust_B_inter','rmb_invoicing_w'))
    except: pass
# (C) asinh
if 'rmb_inv_asinh' in df.columns:
    try:
        m = fit_twfe(df, 'rmb_inv_asinh ~ rmb_infra_it + '+SANC+' + infra_x_sanction + '+CS+' + EntityEffects + TimeEffects')
        rows_rob.append(extract_row(m,'infra_x_sanction','Infra×Sanc [asinh]','robust_C','rmb_inv_asinh'))
    except: pass
# (D) Minimal controls
if 'rmb_invoicing_w' in df.columns:
    try:
        m = fit_twfe(df, 'rmb_invoicing_w ~ rmb_infra_it + '+SANC+' + infra_x_sanction + EntityEffects + TimeEffects')
        rows_rob.append(extract_row(m,'infra_x_sanction','Infra×Sanc [min ctrl]','robust_D','rmb_invoicing_w'))
    except: pass
table_rob = save_table(rows_rob, 'table_robustness')
display(table_rob)

## 11. Temporal Decomposition: Phase 1 vs. Phase 2 (Sanctions Paradox)
Pre-2022 (Phase 1: frustrated demand) vs. post-2022 (Phase 2: active diversification)
subsamples. The 2022 Russia sanctions serve as the empirical Phase 1→2 trigger.

In [ ]:
# ================================================================
# CELL 9B: Phase 1 Subperiod Analysis + Updated Force Vector
# ================================================================
# Purpose: Resolve the Phase 1 time-window inconsistency.
# Table 8.5 uses 2010-2021; Figure 5 uses 2015-2021 Z-scores.
# The 2015-2021 interaction coefficient differs by ~91% from
# 2010-2021 due to pre-CIPS dilution. Both are reported.
# ================================================================
print('='*70)
print('PHASE 1 SUBPERIOD ANALYSIS: Pre-CIPS Dilution Effect')
print('='*70)

FORMULA_INT = ('rmb_invoicing_w ~ rmb_infra_it + ' + SANC
               + ' + infra_x_sanction + ' + CS
               + ' + EntityEffects + TimeEffects')
FORMULA_INT_ASINH = ('rmb_inv_asinh ~ rmb_infra_it + ' + SANC
                     + ' + infra_x_sanction + ' + CS
                     + ' + EntityEffects + TimeEffects')

# --- 1. Estimate interaction for three Phase 1 windows ---
windows = [
    ('2010-2021 (full Phase 1)', df['year'] <= 2021),
    ('2015-2021 (CIPS era)',     (df['year'] >= 2015) & (df['year'] <= 2021)),
    ('2022-2023 (Phase 2)',      df['year'] >= 2022),
]

# 【修改1】在控制台打印的表头中加入 R2
print(f'\n{"Window":<30} {"DV":<10} {"Coeff":>8} {"SE":>8} {"p":>8} {"N":>6} {"N_ctry":>6} {"R2":>6}')
print('-' * 88)

results_for_table = []
for label, mask in windows:
    sub = df[mask].copy()
    for dv, dvname in [(FORMULA_INT, 'Wins.'), (FORMULA_INT_ASINH, 'Asinh')]:
        try:
            m = fit_twfe(sub, dv)
            b = m.params['infra_x_sanction']
            se = m.std_errors['infra_x_sanction']
            p = m.pvalues['infra_x_sanction']
            n = int(m.nobs)

            # 【修改2】安全提取 R方 值。优先提取组内 R方 (rsquared_within)，若无则取整体 R方 (rsquared)
            if hasattr(m, 'rsquared_within'):
                r2 = m.rsquared_within
            elif hasattr(m, 'rsquared'):
                r2 = m.rsquared
            else:
                r2 = float('nan') # 如果模型对象没有R方属性，则填充空值避免报错

            # Count unique countries in regression sample
            cols_needed = ['rmb_invoicing_w' if 'invoicing_w' in dv else 'rmb_inv_asinh',
                           'rmb_infra_it', SANC, 'infra_x_sanction'] + CONTROLS
            sub_clean = sub.dropna(subset=[c for c in cols_needed if c in sub.columns])
            n_ctry = sub_clean['iso3'].nunique()
            stars = '***' if p < 0.01 else '**' if p < 0.05 else '*' if p < 0.10 else ''

            # 【修改3】在控制台输出中打印 R2
            print(f'{label:<30} {dvname:<10} {b:>+8.3f} {se:>8.3f} {p:>8.4f}{stars:>4} {n:>6} {n_ctry:>6} {r2:>6.3f}')

            # 【修改4】将 'r2': r2 存入结果字典中
            results_for_table.append({
                'window': label, 'dv': dvname, 'beta': b, 'se': se, 'p': p,
                'n': n, 'n_ctry': n_ctry, 'r2': r2
            })

            # Also get sanctions main effect
            b_s = m.params[SANC]
            se_s = m.std_errors[SANC]
            p_s = m.pvalues[SANC]
            results_for_table.append({
                'window': label, 'dv': dvname + ' (sanctions main)',
                'beta': b_s, 'se': se_s, 'p': p_s, 'n': n, 'n_ctry': n_ctry, 'r2': r2
            })
        except Exception as e:
            print(f'{label:<30} {dvname:<10} FAILED: {e}')

# --- 2. Compute amplification ratios ---
print('\n' + '='*70)
print('AMPLIFICATION RATIOS')
print('='*70)

# Extract key coefficients
betas = {r['window'][:9]: r['beta'] for r in results_for_table
         if r['dv'] == 'Wins.' and 'sanctions' not in r['dv']}

b_2010 = [r['beta'] for r in results_for_table
          if r['window'].startswith('2010') and r['dv'] == 'Wins.'][0]
b_2015 = [r['beta'] for r in results_for_table
          if r['window'].startswith('2015') and r['dv'] == 'Wins.'][0]
b_p2   = [r['beta'] for r in results_for_table
          if r['window'].startswith('2022') and r['dv'] == 'Wins.'][0]

ratio_full = b_p2 / b_2010
ratio_cips = b_p2 / b_2015

print(f'Phase 2 / Full Phase 1 (2010-2021):  {b_p2:.2f} / {b_2010:.3f} = {ratio_full:.1f}x')
print(f'Phase 2 / CIPS-era Phase 1 (2015-2021): {b_p2:.2f} / {b_2015:.3f} = {ratio_cips:.1f}x')
print(f'\n→ Report as: "approximately {ratio_cips:.0f}- to {ratio_full:.0f}-fold amplification"')

# --- 3. Dilution diagnostic ---
print('\n' + '='*70)
print('DILUTION DIAGNOSTIC')
print('='*70)
print(f'2010-2021 interaction: {b_2010:+.3f}')
print(f'2015-2021 interaction: {b_2015:+.3f}')
print(f'Difference: {(b_2015/b_2010 - 1)*100:.1f}%')
print(f'\nExplanation: In 2010-2014, CIPS did not exist and rmb_infra_it ≈ 0')
print(f'for most countries. The interaction term (infra × sanctions) has')
print(f'near-zero variation in those 5 years, diluting the coefficient.')
print(f'The 2015-2021 estimate is more informative about the actual')
print(f'Push × Pull mechanism during the CIPS era.')

# Check infra variation pre/post CIPS
pre_cips = df[df['year'] < 2015]['infra_x_sanction']
post_cips = df[(df['year'] >= 2015) & (df['year'] <= 2021)]['infra_x_sanction']
print(f'\ninfra_x_sanction variance:')
print(f'  2010-2014: {pre_cips.var():.6f}  (mean={pre_cips.mean():.4f})')
print(f'  2015-2021: {post_cips.var():.6f}  (mean={post_cips.mean():.4f})')
print(f'  Ratio: {post_cips.var()/max(pre_cips.var(), 1e-10):.1f}x more variation in CIPS era')

# --- 4. Updated Force Vector Figure ---
print('\n' + '='*70)
print('UPDATED FORCE VECTOR DIAGRAM')
print('='*70)

# Z-scores: computed from panel means over specified windows
# Phase 1: 2015-2021 (matches the coefficient window now)
p1_data = df[(df['year'] >= 2015) & (df['year'] <= 2021)]
p2_data = df[df['year'] >= 2022]
all_data = df.copy()

# Compute Z-scores relative to full-panel mean and SD
push_mean = df.groupby('year')[SANC].mean()
pull_mean = df.groupby('year')['rmb_infra_it'].mean()
push_overall_mean, push_overall_std = push_mean.mean(), push_mean.std()
pull_overall_mean, pull_overall_std = pull_mean.mean(), pull_mean.std()

p1_years = range(2015, 2022)
p2_years = range(2022, 2024)

P1_PUSH_Z_new = (push_mean.loc[list(p1_years)].mean() - push_overall_mean) / push_overall_std
P1_PULL_Z_new = (pull_mean.loc[list(p1_years)].mean() - pull_overall_mean) / pull_overall_std
P2_PUSH_Z_new = (push_mean.loc[list(p2_years)].mean() - push_overall_mean) / push_overall_std
P2_PULL_Z_new = (pull_mean.loc[list(p2_years)].mean() - pull_overall_mean) / pull_overall_std

print(f'Z-scores (2015-2021 vs 2022-2023):')
print(f'  Phase 1 Push Z: {P1_PUSH_Z_new:.3f}  (was 0.114)')
print(f'  Phase 1 Pull Z: {P1_PULL_Z_new:.3f}  (was 0.556)')
print(f'  Phase 2 Push Z: {P2_PUSH_Z_new:.3f}  (was 1.840)')
print(f'  Phase 2 Pull Z: {P2_PULL_Z_new:.3f}  (was 1.046)')

# New interaction coefficients for figure
P1_INTER_NEW = b_2015   # was 1.58 (from 2010-2021)
P2_INTER_NEW = b_p2     # unchanged

print(f'\n  Phase 1 interaction: {P1_INTER_NEW:+.3f}  (was +1.576)')
print(f'  Phase 2 interaction: {P2_INTER_NEW:+.3f}  (unchanged)')
print(f'  Amplification: {P2_INTER_NEW/P1_INTER_NEW:.1f}x  (was 13.2x)')

# Angle mapping (illustrative)
# Higher interaction → tighter angle (stronger complementarity)
# Use log-ratio proportionality: angle_ratio = log(P2/P1)
import math
angle_ratio = math.log(P2_INTER_NEW / P1_INTER_NEW)
# Set Phase 2 angle = 35° (strong), then Phase 1 = 35° × angle_ratio
# But cap at reasonable range
P2_ANGLE_DEG = 35
P1_ANGLE_DEG = min(80, P2_ANGLE_DEG * angle_ratio)
print(f'\n  Phase 1 angle: {P1_ANGLE_DEG:.0f}°  (was 70°)')
print(f'  Phase 2 angle: {P2_ANGLE_DEG}°  (unchanged)')

# --- Generate updated figure ---
P1_PUSH_Z = P1_PUSH_Z_new
P1_PULL_Z = P1_PULL_Z_new
P2_PUSH_Z = P2_PUSH_Z_new
P2_PULL_Z = P2_PULL_Z_new
P1_RESIST_Z, P2_RESIST_Z = 0.0, 1.5
P1_ANGLE = np.radians(P1_ANGLE_DEG)
P2_ANGLE = np.radians(P2_ANGLE_DEG)

def make_vecs(push_mag, pull_mag, angle, resist_mag):
    midline = np.radians(45)
    push = push_mag * np.array([np.cos(midline+angle/2), np.sin(midline+angle/2)])
    pull = pull_mag * np.array([np.cos(midline-angle/2), np.sin(midline-angle/2)])
    net = push + pull
    resist = np.array([0.,0.])
    if resist_mag > 0.01 and np.linalg.norm(net) > 0.01:
        resist = -net/np.linalg.norm(net) * resist_mag * 0.5
    return push, pull, net, resist, net+resist

fig, axes = plt.subplots(1, 2, figsize=(15, 6.5))

configs = [
    (axes[0], P1_PUSH_Z, P1_PULL_Z, P1_RESIST_Z, P1_ANGLE,
     'Phase 1: Pre-2022\n(Moderate Push, Building Pull)', '2015–2021',
     f'+{P1_INTER_NEW:.2f}***'),
    (axes[1], P2_PUSH_Z, P2_PULL_Z, P2_RESIST_Z, P2_ANGLE,
     'Phase 2: Post-2022\n(Strong Push, Mature Pull, Active Resist)', '2022–2023',
     f'+{P2_INTER_NEW:.1f}**'),
]

for idx, (ax, pz, plz, rz, ang, title, yr, inter) in enumerate(configs):
    sc = 1.3
    push, pull, net, resist, final = make_vecs(pz*sc, plz*sc, ang, rz*sc)
    origin = np.array([0., 0.])
    ax.set_facecolor('#FAFAFA')

    for vec, col, lw in [(push,'#c0392b',3.5),(pull,'#27ae60',3.5)]:
        ax.annotate('', xy=vec, xytext=origin,
                    arrowprops=dict(arrowstyle='-|>', color=col, lw=lw, mutation_scale=18))
    ax.annotate('', xy=net, xytext=origin,
                arrowprops=dict(arrowstyle='-|>', color='#2c3e50', lw=3, mutation_scale=16, ls='--'))
    if np.linalg.norm(resist)>0.05:
        ax.annotate('', xy=net+resist, xytext=net,
                    arrowprops=dict(arrowstyle='-|>', color='#e67e22', lw=3, mutation_scale=16))
    ax.annotate('', xy=final, xytext=origin,
                arrowprops=dict(arrowstyle='-|>', color='#1a1a2e', lw=2.5, mutation_scale=14))

    pe_w = [pe.withStroke(linewidth=3, foreground='white')]
    ax.text(push[0]*0.3 - 0.15, push[1]*0.3 + 0.08,
            f'PUSH\n(Z={pz:.2f})', fontsize=8.5, fontweight='bold',
            color='#c0392b', ha='center', path_effects=pe_w)
    ax.text(pull[0]*0.7 + 0.05, pull[1]*0.7 - 0.15,
            f'PULL\n(Z={plz:.2f})', fontsize=8.5, fontweight='bold',
            color='#27ae60', ha='center', path_effects=pe_w)
    ax.text(final[0]+0.12, final[1]-0.08,
            'Net de-dollar\nforce', fontsize=7.5, color='#1a1a2e',
            ha='left', fontstyle='italic', path_effects=pe_w)
    if np.linalg.norm(resist) > 0.05:
        mid_r = net + resist*0.5
        ax.text(mid_r[0]+0.18, mid_r[1]+0.05,
                f'RESIST\n(Z={rz:.1f})', fontsize=8.5, fontweight='bold',
                color='#e67e22', ha='center', path_effects=pe_w)

    ax.text(0.95, 0.05, f'Interaction: {inter}\n({"moderate" if idx==0 else "strong"} complementarity)',
            transform=ax.transAxes, fontsize=8, ha='right', va='bottom',
            bbox=dict(boxstyle='round,pad=0.4', facecolor='#E8F4FD', alpha=0.8))
    ax.text(0.05, 0.95, yr, transform=ax.transAxes, fontsize=11,
            fontweight='bold', va='top',
            bbox=dict(boxstyle='round,pad=0.3', facecolor='#2c3e50', alpha=0.85),
            color='white')

    ax.set_xlabel('Infrastructure channel'); ax.set_ylabel('Political risk channel')
    ax.set_title(title, fontweight='bold')
    ax.grid(alpha=0.2, ls=':')
    ax.set_aspect('equal')
    pts = np.array([origin, push, pull, net, final, net+resist])
    pad = 0.4
    ax.set_xlim(pts[:,0].min()-pad, pts[:,0].max()+pad)
    ax.set_ylim(pts[:,1].min()-pad, pts[:,1].max()+pad)

fig.suptitle('Force Composition of De-Dollarization (Data-Calibrated)',
             fontsize=14, fontweight='bold', y=1.02)
fig.tight_layout()
fig.savefig(os.path.join(OUT, 'fig_force_vectors_UPDATED.png'), dpi=300, bbox_inches='tight')
plt.show()
print('  Saved: fig_force_vectors_UPDATED.png')

# --- 5. Summary of values for paper updates ---
print('\n' + '='*70)
print('VALUES TO UPDATE IN PAPER')
print('='*70)
print(f'''
Table 8.5 — ADD new row:
  "2015-2021 (CIPS era)" | Wins. | Infra×Sanctions | +{b_2015:.3f} | [SE from output] | [p from output]

Section 8.5 — CHANGE "13.2-fold" to:
  "The amplification ranges from approximately {ratio_cips:.0f}-fold (comparing the
   CIPS-era Phase 1 estimate of +{b_2015:.2f} to the Phase 2 estimate of +{b_p2:.1f})
   to approximately {ratio_full:.0f}-fold (using the full 2010-2021 Phase 1 estimate
   of +{b_2010:.2f}). The difference reflects pre-CIPS dilution: in 2010-2014, CIPS
   had not yet launched and the interaction term had near-zero variation, attenuating
   the full-period coefficient."

Figure 5 — UPDATE:
  Phase 1 year label: "2015-2021" (consistent with Z-scores and coefficient)
  Phase 1 interaction: "+{b_2015:.2f}***" (was "+1.58***")
  Phase 1 Push Z: {P1_PUSH_Z_new:.2f} (verify matches original)
  Phase 1 Pull Z: {P1_PULL_Z_new:.2f} (verify matches original)
  Phase 1 angle: {P1_ANGLE_DEG:.0f}° (was 70°)

Section 9.4 — UPDATE vector description:
  Phase 1 interaction = +{b_2015:.2f}; Phase 2 interaction = +{b_p2:.1f}

Abstract — CHANGE "approximately 13-fold" to:
  "approximately {ratio_cips:.0f}- to {ratio_full:.0f}-fold"

Appendix Figure 5 description — UPDATE accordingly
''')

# --- 6. Save supplementary table ---
pd.DataFrame(results_for_table).to_csv(
    os.path.join(OUT, 'table_phase1_subperiod.csv'), index=False)
print('Saved: table_phase1_subperiod.csv')

### 11.1 Bootstrap Confidence Interval for Phase 2 Amplification Ratio

In [ ]:
# ================================================================
# CELL 12: Bootstrap Phase 2 Amplification + Leave-One-Out
# ================================================================
print('='*70)
print('BOOTSTRAP: Phase 2 Interaction Amplification Ratio')
print('='*70)

np.random.seed(42)
N_BOOT = 1000

# --- Phase 1 & Phase 2 point estimates ---
p1_sub = df[df['year']<=2021].copy()
p2_sub = df[df['year']>=2022].copy()

def get_interaction_coef(data):
    """Fit Push×Pull and return interaction coefficient."""
    try:
        formula = ('rmb_invoicing_w ~ rmb_infra_it + '+SANC
                   +' + infra_x_sanction + '+CS+' + EntityEffects + TimeEffects')
        m = fit_twfe(data, formula)
        return m.params['infra_x_sanction']
    except:
        return np.nan

b_p1 = get_interaction_coef(p1_sub)
b_p2 = get_interaction_coef(p2_sub)
ratio_point = b_p2 / b_p1 if b_p1 != 0 else np.nan
print(f'Phase 1 interaction: {b_p1:.4f}')
print(f'Phase 2 interaction: {b_p2:.4f}')
print(f'Amplification ratio: {ratio_point:.1f}x')

# --- Bootstrap: resample countries within Phase 2, re-estimate ---
p2_countries = p2_sub['iso3'].unique()
boot_ratios = []
boot_p2_coefs = []

for b in range(N_BOOT):
    # Resample countries with replacement
    sampled = np.random.choice(p2_countries, size=len(p2_countries), replace=True)
    # Build bootstrap sample: for each sampled country, take all its years
    frames = []
    for j, iso in enumerate(sampled):
        tmp = p2_sub[p2_sub['iso3']==iso].copy()
        tmp['iso3'] = f'{iso}_{j}'  # make unique entity ID
        frames.append(tmp)
    if not frames: continue
    boot_df = pd.concat(frames, ignore_index=True)
    coef = get_interaction_coef(boot_df)
    if not np.isnan(coef):
        boot_p2_coefs.append(coef)
        if b_p1 != 0:
            boot_ratios.append(coef / b_p1)

if boot_p2_coefs:
    p2_boot_arr = np.array(boot_p2_coefs)
    ci_lo_p2 = np.percentile(p2_boot_arr, 2.5)
    ci_hi_p2 = np.percentile(p2_boot_arr, 97.5)
    print(f'\nPhase 2 interaction bootstrap 95% CI: [{ci_lo_p2:.2f}, {ci_hi_p2:.2f}]')
    print(f'  (point estimate: {b_p2:.2f})')

    if boot_ratios:
        ratio_arr = np.array(boot_ratios)
        ci_lo_r = np.percentile(ratio_arr, 2.5)
        ci_hi_r = np.percentile(ratio_arr, 97.5)
        print(f'Amplification ratio bootstrap 95% CI: [{ci_lo_r:.1f}x, {ci_hi_r:.1f}x]')
        print(f'  (point estimate: {ratio_point:.1f}x)')
else:
    print('Bootstrap failed — insufficient variation.')

# --- Leave-one-out sensitivity for Phase 2 ---
print('\n--- Leave-One-Out Sensitivity (Phase 2) ---')
loo_results = []
for iso in p2_countries:
    sub_loo = p2_sub[p2_sub['iso3'] != iso].copy()
    coef = get_interaction_coef(sub_loo)
    loo_results.append({'dropped': iso, 'beta_interaction': round(coef, 4) if not np.isnan(coef) else np.nan})

loo_df = pd.DataFrame(loo_results).dropna().sort_values('beta_interaction')
print(f'Full sample: {b_p2:.4f}')
print(f'Range when dropping one country: [{loo_df["beta_interaction"].min():.4f}, '
      f'{loo_df["beta_interaction"].max():.4f}]')
print('\nMost influential countries (largest change from dropping):')
loo_df['delta'] = abs(loo_df['beta_interaction'] - b_p2)
for _, row in loo_df.nlargest(5, 'delta').iterrows():
    print(f'  Drop {row["dropped"]}: beta = {row["beta_interaction"]:.4f} '
          f'(delta = {row["delta"]:+.4f})')

loo_df.to_csv(os.path.join(OUT, 'phase2_leave_one_out.csv'), index=False)
print('  Saved: phase2_leave_one_out.csv')

## 12. Figures

### Figure 1: Event Study — CIPS Effect on Clearing-Bank Adoption

In [ ]:
# ================================================================
# CELL 13: FIGURE — Three Forces (Push-Pull-Resist)
# ================================================================
print('='*70); print('FIGURE: Three Forces'); print('='*70)

STABLECOIN = {2010:0,2011:0,2012:0,2013:0,2014:0.001,2015:0.005,
              2016:0.01,2017:0.02,2018:2.7,2019:5.7,2020:28.4,
              2021:155.6,2022:138.5,2023:128.8}
CBDC_PILOT = {2010:0,2011:0,2012:0,2013:0,2014:0,2015:0,
              2016:0,2017:0,2018:1,2019:2,2020:5,
              2021:14,2022:20,2023:36}
STABLECOIN_REG = {2010:0,2011:0,2012:0,2013:0,2014:0,2015:0,
                  2016:0,2017:1,2018:2,2019:5,2020:8,
                  2021:15,2022:25,2023:45}

years = list(range(2010, 2024))
push_ts = df.groupby('year')[SANC].agg(['mean','sem']).reindex(years)
pull_ts = df.groupby('year')['rmb_infra_it'].agg(['mean','sem']).reindex(years)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# (A) Push
ax = axes[0,0]
ax.fill_between(years, push_ts['mean']-1.96*push_ts['sem'],
                push_ts['mean']+1.96*push_ts['sem'], alpha=0.2, color='#c0392b')
ax.plot(years, push_ts['mean'], 'o-', color='#c0392b', lw=2.5, ms=5)
ax.axvline(2014, color='gray', ls=':', alpha=0.5, label='Crimea')
ax.axvline(2022, color='gray', ls='--', alpha=0.5, label='Russia full')
ax.set_title('Force 1: PUSH\n(Sanctions Intensity)', fontweight='bold')
ax.set_ylabel('Mean sanctions exposure'); ax.set_xlabel('Year')
ax.legend(fontsize=8, loc='upper left')

# (B) Pull — CIPS at 2015
ax = axes[0,1]
ax.fill_between(years, pull_ts['mean']-1.96*pull_ts['sem'],
                pull_ts['mean']+1.96*pull_ts['sem'], alpha=0.2, color='#27ae60')
ax.plot(years, pull_ts['mean'], 's-', color='#27ae60', lw=2.5, ms=5)
ax.axvline(2015, color='gray', ls=':', alpha=0.5, label='CIPS launch (Oct 2015)')
ax.set_title('Force 2: PULL\n(RMB Infrastructure)', fontweight='bold')
ax.set_ylabel('Mean infra index (0-2)'); ax.set_xlabel('Year')
ax.legend(fontsize=8, loc='upper left')

# (C) Resist — fix annotation placement
ax = axes[1,0]; ax2 = ax.twinx()
sc_vals = [STABLECOIN[y] for y in years]
cbdc_vals = [CBDC_PILOT[y] for y in years]
reg_vals = [STABLECOIN_REG[y] for y in years]
ax.bar(years, sc_vals, alpha=0.25, color='#e67e22', width=0.6)
ax.plot(years, sc_vals, 'D-', color='#d35400', lw=2, ms=4, label='Stablecoin mcap ($B)')
ax2.plot(years, cbdc_vals, '^-', color='#8e44ad', lw=2, ms=5, label='CBDC pilot/live')
ax2.plot(years, reg_vals, 'v-', color='#e91e63', lw=1.5, ms=4, label='Stablecoin reg.')
ax.set_title('Force 3: RESIST\n(Digital Dollar Reproduction)', fontweight='bold')
ax.set_ylabel('Stablecoin mcap ($B)'); ax.set_xlabel('Year')
ax2.set_ylabel('Count of countries')
ax.legend(fontsize=7, loc='upper left')
ax2.legend(fontsize=7, loc='center left', bbox_to_anchor=(0.0, 0.55))
# Annotations — placed OUTSIDE data area to avoid overlap
ax.annotate('Tether launch', xy=(2014, 0.001), xytext=(2015.5, 35),
            fontsize=7, color='gray', ha='center',
            arrowprops=dict(arrowstyle='->', color='gray', lw=0.8))
ax.annotate('DeFi summer', xy=(2020, 28.4), xytext=(2018.5, 70),
            fontsize=7, color='gray', ha='center',
            arrowprops=dict(arrowstyle='->', color='gray', lw=0.8))

# (D) Z-score field
ax = axes[1,1]
push_z = (push_ts['mean'] - push_ts['mean'].mean()) / push_ts['mean'].std()
pull_z = (pull_ts['mean'] - pull_ts['mean'].mean()) / pull_ts['mean'].std()
sc_s = pd.Series(sc_vals, index=years)
sc_z = (sc_s - sc_s.mean()) / sc_s.std()
cbdc_s = pd.Series(cbdc_vals, index=years)
cbdc_z = (cbdc_s - cbdc_s.mean()) / (cbdc_s.std() + 1e-10)
ax.plot(years, push_z, 'o-', color='#c0392b', lw=2.5, label='Push: sanctions (Z)')
ax.plot(years, pull_z, 's-', color='#27ae60', lw=2.5, label='Pull: RMB infra (Z)')
ax.plot(years, sc_z, 'D-', color='#ff7f0e', lw=2.5, label='Resist: stablecoins (Z)')
ax.plot(years, cbdc_z, '^--', color='#8e44ad', lw=1.5, label='Resist: CBDC pilots (Z)')
ax.axhline(0, color='black', lw=0.8)
ax.axvline(2015, color='gray', ls=':', alpha=0.5)
ax.axvline(2022, color='gray', ls='--', alpha=0.5)
ax.set_title('Vector Field\n(Standardized Forces)', fontweight='bold')
ax.set_ylabel('Z-score'); ax.set_xlabel('Year')
ax.legend(fontsize=7, loc='upper left')

for a in axes.flat: a.grid(alpha=0.3, ls='--')
fig.tight_layout()
fig.savefig(os.path.join(OUT,'fig_three_forces.png'), dpi=300, bbox_inches='tight')
plt.show()
print('  Saved: fig_three_forces.png')

### Figure 2: Event Study — CIPS Effect on RMB Invoicing Share

In [ ]:
# ================================================================
# CELL 14: FIGURE — Three-Role Trajectories
# ================================================================
print('='*70); print('FIGURE: Three-Role Framework'); print('='*70)
CR = {'Sender':'#2980b9','Target':'#c0392b','Observer':'#27ae60'}
MK = {'Sender':'D','Target':'o','Observer':'s'}

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# (a) Clearing Bank
ax = axes[0,0]
for role in ['Sender','Target','Observer']:
    ts = df[df['role']==role].groupby('year')['clearing_bank_it'].mean()
    ax.plot(ts.index, ts, MK[role]+'-', color=CR[role], lw=2, ms=5, label=role)
    if role=='Observer':
        se = df[df['role']==role].groupby('year')['clearing_bank_it'].sem()
        ax.fill_between(ts.index, ts-1.96*se, ts+1.96*se, alpha=0.15, color=CR[role])
ax.axvline(2015, color='gray', ls=':', alpha=0.5)
ax.text(2015.3, 0.12, 'CIPS\nlaunch', fontsize=7, color='gray')
ax.set_title('(a) Clearing Bank Adoption', fontweight='bold')
ax.set_ylabel('Share of countries'); ax.legend(fontsize=9)

# (b) Sanctions
ax = axes[0,1]
for role in ['Sender','Target','Observer']:
    ts = df[df['role']==role].groupby('year')[SANC].mean()
    ax.plot(ts.index, ts, MK[role]+'-', color=CR[role], lw=2, ms=5, label=role)
ax.axvline(2022, color='gray', ls='--', alpha=0.5)
ax.text(2022.3, ax.get_ylim()[1]*0.85 if ax.get_ylim()[1]>0 else 0.3,
        'Russia\n2022', fontsize=7, color='gray')
ax.set_title('(b) Sanctions Exposure', fontweight='bold')
ax.set_ylabel('Mean sanctions index'); ax.legend(fontsize=9)

# (c) RMB Invoicing
ax = axes[1,0]
for role in ['Sender','Target','Observer']:
    sub = df[(df['role']==role) & df['rmb_invoicing_w'].notna()]
    if len(sub)==0: continue
    ts = sub.groupby('year')['rmb_invoicing_w'].mean()
    ax.plot(ts.index, ts, MK[role]+'-', color=CR[role], lw=2, ms=5, label=role)
    if role=='Observer':
        se = sub.groupby('year')['rmb_invoicing_w'].sem()
        ax.fill_between(ts.index, ts-1.96*se, ts+1.96*se, alpha=0.15, color=CR[role])
ax.set_title('(c) RMB Invoicing Share', fontweight='bold')
ax.set_ylabel('Mean invoicing (%)'); ax.legend(fontsize=9)

# (d) Rolling interaction
ax = axes[1,1]
windows = []
for end_yr in range(2014, 2024):
    start_yr = max(2010, end_yr-4)
    sub = df[(df['year']>=start_yr)&(df['year']<=end_yr)].copy()
    try:
        m = fit_twfe(sub, 'rmb_invoicing_w ~ rmb_infra_it + '+SANC+' + infra_x_sanction + '+CS+' + EntityEffects + TimeEffects')
        windows.append({'year':end_yr, 'beta':m.params['infra_x_sanction'],
                        'se':m.std_errors['infra_x_sanction']})
    except: pass
if windows:
    wdf = pd.DataFrame(windows)
    ax.plot(wdf['year'], wdf['beta'], 's-', color='#8e44ad', lw=2, ms=6)
    ax.fill_between(wdf['year'], wdf['beta']-1.96*wdf['se'],
                    wdf['beta']+1.96*wdf['se'], alpha=0.2, color='#8e44ad')
    ax.axhline(0, color='black', lw=0.8)
    ax.axvline(2022, color='gray', ls='--', alpha=0.5)
ax.set_title('(d) Push × Pull Interaction\n(5-year rolling window)', fontweight='bold')
ax.set_ylabel('Coefficient on Infra × Sanctions'); ax.set_xlabel('Year')

for a in axes.flat: a.grid(alpha=0.3, ls='--')
fig.tight_layout()
fig.savefig(os.path.join(OUT,'fig_three_role.png'), dpi=300, bbox_inches='tight')
plt.show()
print('  Saved: fig_three_role.png')

### Figure 3: CS-DiD Group-Time ATT Estimates

In [ ]:
# ================================================================
# CELL 15: FIGURE — Observer Quadrant (Heatmap + Bar)
# ================================================================
print('='*70); print('FIGURE: Observer Quadrant (unified sample)'); print('='*70)

obs_df = df[df['role']=='Observer'].copy()
cs = obs_df.groupby('iso3').agg({
    SANC: 'mean', 'rmb_infra_it': 'mean', 'rmb_invoicing_w': 'mean'
}).dropna()   # Unified filter: drop NaN on ALL three columns

cs['has_sanction'] = (cs[SANC] > 0).astype(int)
cs['has_infra'] = (cs['rmb_infra_it'] > 0).astype(int)
cs['region'] = cs.index.map(lambda x: REGION_MAP.get(x,'Other'))

# Print Ns for verification
for s,i,lab in [(1,0,'Q1'),(1,1,'Q2'),(0,0,'Q3'),(0,1,'Q4')]:
    n = len(cs[(cs['has_sanction']==s)&(cs['has_infra']==i)])
    m = cs[(cs['has_sanction']==s)&(cs['has_infra']==i)]['rmb_invoicing_w'].mean()
    print(f'  {lab}: N={n}, mean={m:.3f}')

fig, axes = plt.subplots(1, 2, figsize=(13, 5),
                         gridspec_kw={'width_ratios': [1, 1.4]})

# (a) Heatmap
ax = axes[0]
matrix = np.zeros((2,2)); labels_m = np.empty((2,2), dtype=object)
counts_m = np.empty((2,2), dtype=object)
for si,sanc in enumerate([1,0]):
    for ii,infra in enumerate([0,1]):
        sub = cs[(cs['has_sanction']==sanc)&(cs['has_infra']==infra)]
        matrix[si,ii] = sub['rmb_invoicing_w'].mean()
        labels_m[si,ii] = f'{matrix[si,ii]:.3f}'
        counts_m[si,ii] = f'N={len(sub)}'
im = ax.imshow(matrix, cmap='YlOrRd', aspect='auto', vmin=0, vmax=1.0)
for i in range(2):
    for j in range(2):
        tc = 'white' if matrix[i,j]>0.5 else 'black'
        ax.text(j, i-0.12, labels_m[i,j], ha='center', va='center',
                fontsize=16, fontweight='bold', color=tc)
        ax.text(j, i+0.18, counts_m[i,j], ha='center', va='center',
                fontsize=10, color=tc, alpha=0.8)
ax.set_xticks([0,1])
ax.set_xticklabels(['No Infrastructure\n(Pull = 0)','Has Infrastructure\n(Pull > 0)'])
ax.set_yticks([0,1])
ax.set_yticklabels(['Sanctioned\n(Push > 0)','Not Sanctioned\n(Push = 0)'])
ax.set_title('(a) Mean RMB Invoicing by Quadrant', fontweight='bold')
plt.colorbar(im, ax=ax, shrink=0.7, label='Mean invoicing share (%)')

# (b) Bar chart
ax = axes[1]
qd = []
for s,i,lab,col in [(1,0,'Q1: Sanctioned\nNo Infra','#e74c3c'),
                     (1,1,'Q2: Sanctioned\n+ Infra','#9b59b6'),
                     (0,0,'Q3: Not Sanctioned\nNo Infra','#bdc3c7'),
                     (0,1,'Q4: Not Sanctioned\n+ Infra','#27ae60')]:
    sub = cs[(cs['has_sanction']==s)&(cs['has_infra']==i)]
    qd.append({'label':lab,'mean':sub['rmb_invoicing_w'].mean(),
               'se':sub['rmb_invoicing_w'].sem(),'n':len(sub),'color':col})
qdf = pd.DataFrame(qd)
ax.bar(range(len(qdf)), qdf['mean'], yerr=1.96*qdf['se'], capsize=5,
       color=qdf['color'], alpha=0.8, edgecolor='white', lw=1.5)
ax.set_xticks(range(len(qdf))); ax.set_xticklabels(qdf['label'], fontsize=9)
ax.set_ylabel('Mean RMB Invoicing Share (%)')
ax.set_title('(b) Invoicing by Quadrant (with 95% CI)', fontweight='bold')
for i,row in qdf.iterrows():
    ax.text(i, row['mean']+1.96*row['se']+0.02, f'N={row["n"]}',
            ha='center', fontsize=9, color='#2c3e50')
# Q1→Q2 multiplier
q1m, q2m = qdf.loc[0,'mean'], qdf.loc[1,'mean']
if q1m > 0:
    mult = q2m/q1m
    mid_y = (q1m+q2m)/2
    ax.annotate('', xy=(1, q2m*0.9), xytext=(0, q1m*1.1),
                arrowprops=dict(arrowstyle='->', color='#8e44ad', lw=2))
    ax.text(0.5, mid_y+0.03, f'{mult:.1f}x', fontsize=13, fontweight='bold',
            color='#8e44ad', ha='center')

fig.tight_layout()
fig.savefig(os.path.join(OUT,'fig_observer_heatmap_bar.png'), dpi=300, bbox_inches='tight')
plt.show()
print('  Saved: fig_observer_heatmap_bar.png')

### Figure 4: Phase 1 vs. Phase 2 Temporal Decomposition

In [ ]:
# ================================================================
# CELL 16: FIGURE — Force Vector Diagram (fixed labels)
# ================================================================
print('='*70); print('FIGURE: Force Vectors'); print('='*70)

P1_PUSH_Z, P2_PUSH_Z = 0.114, 1.840
P1_PULL_Z, P2_PULL_Z = 0.556, 1.046
P1_RESIST_Z, P2_RESIST_Z = 0.0, 1.5
P1_ANGLE, P2_ANGLE = np.radians(70), np.radians(35)

def make_vecs(push_mag, pull_mag, angle, resist_mag):
    midline = np.radians(45)
    push = push_mag * np.array([np.cos(midline+angle/2), np.sin(midline+angle/2)])
    pull = pull_mag * np.array([np.cos(midline-angle/2), np.sin(midline-angle/2)])
    net = push + pull
    resist = np.array([0.,0.])
    if resist_mag > 0.01 and np.linalg.norm(net) > 0.01:
        resist = -net/np.linalg.norm(net) * resist_mag * 0.5
    return push, pull, net, resist, net+resist

fig, axes = plt.subplots(1, 2, figsize=(15, 6.5))

configs = [
    (axes[0], P1_PUSH_Z, P1_PULL_Z, P1_RESIST_Z, P1_ANGLE,
     'Phase 1: Pre-2022\n(Moderate Push, Building Pull)', 'Pre-2022', '+1.58***'),
    (axes[1], P2_PUSH_Z, P2_PULL_Z, P2_RESIST_Z, P2_ANGLE,
     'Phase 2: Post-2022\n(Strong Push, Mature Pull, Active Resist)', '2022-2023', '+20.8**'),
]

for idx, (ax, pz, plz, rz, ang, title, yr, inter) in enumerate(configs):
    sc = 1.3
    push, pull, net, resist, final = make_vecs(pz*sc, plz*sc, ang, rz*sc)
    origin = np.array([0., 0.])
    ax.set_facecolor('#FAFAFA')

    # Draw arrows
    for vec, col, lw in [(push,'#c0392b',3.5),(pull,'#27ae60',3.5)]:
        ax.annotate('', xy=vec, xytext=origin,
                    arrowprops=dict(arrowstyle='-|>', color=col, lw=lw, mutation_scale=18))
    ax.annotate('', xy=net, xytext=origin,
                arrowprops=dict(arrowstyle='-|>', color='#2c3e50', lw=3, mutation_scale=16, ls='--'))
    if np.linalg.norm(resist)>0.05:
        ax.annotate('', xy=net+resist, xytext=net,
                    arrowprops=dict(arrowstyle='-|>', color='#e67e22', lw=3, mutation_scale=16))
    ax.annotate('', xy=final, xytext=origin,
                arrowprops=dict(arrowstyle='-|>', color='#1a1a2e', lw=2.5, mutation_scale=14))

    # Labels — use text with white outline for readability, offset from arrows
    pe_w = [pe.withStroke(linewidth=3, foreground='white')]
    # PUSH label: offset to upper-left of arrow tip
    ax.text(push[0]*0.3 - 0.15, push[1]*0.3 + 0.08,
            f'PUSH\n(Z={pz:.2f})', fontsize=8.5, fontweight='bold',
            color='#c0392b', ha='center', path_effects=pe_w)
    # PULL label: offset to lower-right of arrow tip
    ax.text(pull[0]*0.7 + 0.05, pull[1]*0.7 - 0.15,
            f'PULL\n(Z={plz:.2f})', fontsize=8.5, fontweight='bold',
            color='#27ae60', ha='center', path_effects=pe_w)
    # Net label
    ax.text(final[0]+0.12, final[1]-0.08,
            'Net de-dollar\nforce', fontsize=7.5, color='#1a1a2e',
            ha='left', fontstyle='italic', path_effects=pe_w)
    # Resist label
    if np.linalg.norm(resist) > 0.05:
        mid_r = net + resist*0.5
        ax.text(mid_r[0]+0.18, mid_r[1]+0.05,
                f'RESIST\n(Z={rz:.1f})', fontsize=8.5, fontweight='bold',
                color='#e67e22', ha='center', path_effects=pe_w)

    # Interaction box
    ax.text(0.95, 0.05, f'Interaction: {inter}\n({"moderate" if idx==0 else "strong"} complementarity)',
            transform=ax.transAxes, fontsize=8, ha='right', va='bottom',
            bbox=dict(boxstyle='round,pad=0.4', facecolor='#E8F4FD', alpha=0.8))
    # Year badge
    ax.text(0.05, 0.95, yr, transform=ax.transAxes, fontsize=11,
            fontweight='bold', va='top',
            bbox=dict(boxstyle='round,pad=0.3', facecolor='#2c3e50', alpha=0.85),
            color='white')

    ax.set_xlabel('Infrastructure channel'); ax.set_ylabel('Political risk channel')
    ax.set_title(title, fontweight='bold')
    ax.grid(alpha=0.2, ls=':')
    ax.set_aspect('equal')
    # Axis limits with padding
    pts = np.array([origin, push, pull, net, final, net+resist])
    pad = 0.4
    ax.set_xlim(pts[:,0].min()-pad, pts[:,0].max()+pad)
    ax.set_ylim(pts[:,1].min()-pad, pts[:,1].max()+pad)

fig.suptitle('Force Composition of De-Dollarization (Data-Calibrated)',
             fontsize=14, fontweight='bold', y=1.02)
fig.tight_layout()
fig.savefig(os.path.join(OUT,'fig_force_vectors.png'), dpi=300, bbox_inches='tight')
plt.show()
print('  Saved: fig_force_vectors.png')

### Figure 5: Quadrant Analysis — Sanctions Exposure × RMB Infrastructure

In [ ]:
# ================================================================
# CELL 17: FIGURE — Observer Quadrant Strip Chart (Appendix)
#          FIXED: whitespace eliminated, proper axis sizing
# ================================================================
print('='*70); print('FIGURE: Observer Quadrant Strip (Appendix)'); print('='*70)

region_colors = {'Africa':'#e74c3c','Asia':'#3498db','Europe':'#27ae60',
                 'Americas':'#f39c12','Oceania':'#9b59b6'}

quadrants = [
    (1, 0, 'Q1: Sanctioned, No Infra\n(Frustrated Demand)', '#fadbd8'),
    (1, 1, 'Q2: Sanctioned + Infra\n(Activation Zone)', '#e8daef'),
    (0, 0, 'Q3: No Sanctions, No Infra\n(Status Quo)', '#f2f4f4'),
    (0, 1, 'Q4: No Sanctions + Infra\n(Voluntary Adoption)', '#d5f5e3'),
]

# Compute x-axis max from actual data
max_inv = max(cs['rmb_invoicing_w'].quantile(0.99), 2.0)

# Use constrained_layout and explicit figure size to avoid whitespace
fig, axarr = plt.subplots(2, 2, figsize=(12, 8), constrained_layout=True)
# Map: row0=sanctioned (top), row1=not sanctioned (bottom)
#       col0=no infra (left), col1=has infra (right)
panel_map = {(1,0):(0,0), (1,1):(0,1), (0,0):(1,0), (0,1):(1,1)}

for sanc, infra, title, bg_color in quadrants:
    r, c = panel_map[(sanc, infra)]
    ax = axarr[r, c]
    ax.set_facecolor(bg_color)

    sub = cs[(cs['has_sanction']==sanc)&(cs['has_infra']==infra)].copy()
    sub = sub.sort_values('rmb_invoicing_w', ascending=True)
    if len(sub)==0: continue

    ypos = np.arange(len(sub))
    colors = [region_colors.get(r2,'#95a5a6') for r2 in sub['region']]
    ax.barh(ypos, sub['rmb_invoicing_w'], color=colors, alpha=0.7,
            height=0.7, edgecolor='white', lw=0.3)

    # Label top countries
    top_idx = sub.nlargest(min(6, len(sub)), 'rmb_invoicing_w').index
    for j, (iso, rd) in enumerate(sub.iterrows()):
        if iso in top_idx or rd['rmb_invoicing_w'] > 0.3:
            ax.text(rd['rmb_invoicing_w']+max_inv*0.015, ypos[j], iso,
                    fontsize=6, va='center', fontweight='bold', color='#2c3e50')

    mean_inv = sub['rmb_invoicing_w'].mean()
    ax.axvline(mean_inv, color='#2c3e50', ls='--', lw=1.2, alpha=0.7)
    ax.text(0.02, 0.97, f'mean={mean_inv:.2f}', transform=ax.transAxes,
            fontsize=8, va='top', fontweight='bold',
            bbox=dict(facecolor='white', alpha=0.7, boxstyle='round,pad=0.2'))
    ax.set_xlim(0, max_inv)
    ax.set_yticks([])
    ax.set_xlabel('Mean RMB Invoicing Share (%)')
    ax.set_title(title, fontsize=10, fontweight='bold',
                 color='#c0392b' if sanc==1 else '#27ae60')
    ax.text(0.98, 0.02, f'N = {len(sub)}', transform=ax.transAxes,
            ha='right', va='bottom', fontsize=9, color='#7f8c8d')

fig.suptitle('Observer Countries: Push × Pull Quadrant Analysis',
             fontsize=13, fontweight='bold')

# Add manual legend below
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor=c, label=r) for r,c in region_colors.items()]
fig.legend(handles=legend_elements, loc='lower center', ncol=5,
           fontsize=9, frameon=True, bbox_to_anchor=(0.5, -0.02))

fig.savefig(os.path.join(OUT,'fig_observer_quadrants_panel.png'),
            dpi=300, bbox_inches='tight')
plt.show()
print('  Saved: fig_observer_quadrants_panel.png')

# 在 v29 notebook CELL 17 末尾追加
print('='*70)
print('Table 4.1 trajectory column: RMB invoicing by role × year')
print('='*70)
traj = df.groupby(['role','year'])['rmb_invoicing_w'].mean().unstack()
print(traj.round(3))

# 三角色2010 → 2023 端点
print('\n2010 vs 2023 endpoints:')
for role in ['Sender','Target','Observer']:
    sub = df[df['role']==role]
    v_2010 = sub[sub['year']==2010]['rmb_invoicing_w'].mean()
    v_2022 = sub[sub['year']==2022]['rmb_invoicing_w'].mean()
    v_2023 = sub[sub['year']==2023]['rmb_invoicing_w'].mean()
    print(f'  {role}: 2010={v_2010:.3f}, 2022={v_2022:.3f}, 2023={v_2023:.3f}')

## 13. Paper Statistics Verification

In [ ]:
# ---- Verify key statistics mentioned in the paper ----
print('=== PAPER STATISTICS VERIFICATION ===')

# 1. Zero share in DV
zero_pct = (df['rmb_invoicing_share'] == 0).sum() / df['rmb_invoicing_share'].notna().sum()
print(f'DV exact zeros: {zero_pct:.1%} (paper says 54%)')

# 2. Within-country variance share
from scipy import stats
total_var = df['rmb_invoicing_share'].dropna().var()
country_means = df.groupby('iso3')['rmb_invoicing_share'].transform('mean')
within_var = (df['rmb_invoicing_share'] - country_means).dropna().var()
print(f'Within-country variance share: {within_var/total_var:.1%} (paper says 28%)')

# 3. Phase 2 sample size
p2_inv = df[(df['year']>=2022) & (df['rmb_invoicing_w'].notna())]
print(f'Phase 2 N (with invoicing): {len(p2_inv)} (paper says 188)')
print(f'Phase 2 unique countries: {p2_inv["iso3"].nunique()} (paper says ~94)')

# 4. CIPS cohort sizes
cohorts = df[df['CIPSit']==1].groupby('iso3')['year'].min().value_counts().sort_index()
print(f'CIPS cohort sizes: {dict(cohorts)} (paper says 2015:8, 2016:32, 2018:1)')

# 5. Phase 1 interaction coefficient (2015-2021 only vs 2010-2021)
formula_int = ('rmb_invoicing_w ~ rmb_infra_it + ' + SANC +
               ' + infra_x_sanction + ' + CS + ' + EntityEffects + TimeEffects')
m_10_21 = fit_twfe(df[df['year']<=2021], formula_int)
m_15_21 = fit_twfe(df[(df['year']>=2015) & (df['year']<=2021)], formula_int)
b_10 = m_10_21.params['infra_x_sanction']
b_15 = m_15_21.params['infra_x_sanction']
print(f'Interaction 2010-2021: {b_10:.3f} (paper says +1.576)')
print(f'Interaction 2015-2021: {b_15:.3f} (for Figure 5 consistency check)')
print(f'  Difference: {abs(b_10-b_15)/abs(b_10):.1%}')
print(f'  → If <10%, use Solution A (text clarification)')
print(f'  → If >10%, use Solution B (re-estimate for figure)')

## 14. Output Export Summary

In [ ]:
# ================================================================
# CELL 18: Export Summary
# ================================================================
import glob
print('='*70); print('ALL OUTPUTS'); print('='*70)
outputs = sorted(glob.glob(os.path.join(OUT, '*')))
for f in outputs:
    print(f'  {os.path.basename(f):45s} {os.path.getsize(f):>8,d} bytes')

print(f'\nTotal files: {len(outputs)}')
print('\n--- THESIS INTEGRATION NOTES ---')
print('1. All CIPS annotations at 2015 (Oct 8, 2015 = Phase 1 launch)')
print('2. Quadrant Ns are UNIFIED (same dropna filter for heatmap and bar)')
print(f'   N change from v27→v29: countries with NaN invoicing now excluded')
print('3. CS-DiD: parallel trends validated at t<=-3; t=-2,-1 = anticipation')
print('4. Phase 2 bootstrap + leave-one-out sensitivity added')
print('5. PPML interaction insignificance explained (exponential link)')
print('6. IV: weak instrument (F=1.8), appendix only')
print('7. All interaction coefficients are CONSERVATIVE LOWER BOUNDS')
print('   (dollar-invoiced, RMB-settled hybrid mode in Russia/Gulf corridors)')

---
## 15. Supplementary Analyses

Additional diagnostics that strengthen the core results.

| Cell | Analysis | Purpose |
|------|----------|---------|
| 15.1 | Argentina Deep Dive | Phase 2 leverage point — DFBETA diagnostics |
| 15.2 | Marginal Effects Plot | ∂RMB/∂Sanctions as continuous f(Infrastructure) |
| 15.3 | Permutation Inference | Non-parametric Phase 2 test (1,000 permutations) |
| 15.4 | Placebo Treatment Date | Falsification: CIPS shifted 3 years earlier |
| 15.5 | Force Composition Figure | Integrated Push–Pull–Resist vector diagram |


### 15.1 Argentina Deep Dive — Phase 2 Leverage Point

In [ ]:
# ================================================================
# CELL 1: ARGENTINA DEEP DIVE
# ================================================================
print('='*70)
print('ARGENTINA DEEP DIVE: Why is ARG the Phase 2 leverage point?')
print('='*70)

# --- 1a: Show Argentina's raw data ---
arg = df[df['iso3']=='ARG'].sort_values('year')
cols_show = ['year','rmb_invoicing_w','rmb_infra_it','clearing_bank_it',
             'swap_line_it','CIPSit',SANC,'infra_x_sanction']
cols_avail = [c for c in cols_show if c in arg.columns]
print('\nArgentina panel data:')
print(arg[cols_avail].to_string(index=False))

# --- 1b: Compare ARG to panel averages in Phase 2 ---
p2 = df[df['year']>=2022].copy()
print('\n--- Phase 2 (2022-2023) Comparison ---')
for v in ['rmb_invoicing_w','rmb_infra_it',SANC,'infra_x_sanction']:
    if v not in p2.columns: continue
    arg_val = p2[p2['iso3']=='ARG'][v].mean()
    panel_mean = p2[v].mean()
    panel_sd = p2[v].std()
    z = (arg_val - panel_mean) / panel_sd if panel_sd > 0 else 0
    print(f'  {v:35s}: ARG={arg_val:.4f}  Panel mean={panel_mean:.4f}  Z={z:+.2f}')

# --- 1c: Phase 2 regression WITH and WITHOUT Argentina ---
formula = 'rmb_invoicing_w ~ rmb_infra_it + '+SANC+' + infra_x_sanction + '+CS+' + EntityEffects + TimeEffects'

print('\n--- Phase 2 Regression: Full vs. Excluding ARG ---')
m_full = fit_twfe(p2, formula)
m_no_arg = fit_twfe(p2[p2['iso3']!='ARG'], formula)

for label, m in [('Full Phase 2', m_full), ('Excl. Argentina', m_no_arg)]:
    b = m.params['infra_x_sanction']
    se = m.std_errors['infra_x_sanction']
    p_val = m.pvalues['infra_x_sanction']
    stars = '***' if p_val<0.01 else '**' if p_val<0.05 else '*' if p_val<0.1 else ''
    print(f'  {label:20s}: beta={b:+.3f} (SE={se:.3f}, p={p_val:.4f}) {stars}  N={int(m.nobs)}')

# --- 1d: DFBETAS for interaction term ---
print('\n--- Influence Diagnostic: Top 5 DFBETAS for infra_x_sanction ---')
# Manual DFBETAS: for each country, compute (beta_full - beta_drop_i) / SE_full
b_full = m_full.params['infra_x_sanction']
se_full = m_full.std_errors['infra_x_sanction']
dfbetas = []
for iso in p2['iso3'].unique():
    sub = p2[p2['iso3']!=iso]
    try:
        m_i = fit_twfe(sub, formula)
        b_i = m_i.params['infra_x_sanction']
        dfbeta = (b_full - b_i) / se_full
        dfbetas.append({'iso3':iso, 'dfbeta':dfbeta, 'beta_excl':b_i})
    except:
        pass

dfb_df = pd.DataFrame(dfbetas).sort_values('dfbeta', key=abs, ascending=False)
print(f'  Threshold (2/sqrt(N)): {2/np.sqrt(len(p2["iso3"].unique())):.3f}')
for _, row in dfb_df.head(10).iterrows():
    flag = ' ⚠ INFLUENTIAL' if abs(row['dfbeta']) > 2/np.sqrt(len(p2['iso3'].unique())) else ''
    print(f'  {row["iso3"]}: DFBETA={row["dfbeta"]:+.3f}, beta_excl={row["beta_excl"]:+.3f}{flag}')

dfb_df.to_csv(os.path.join(OUT, 'phase2_dfbetas.csv'), index=False)
print('\n  Saved: phase2_dfbetas.csv')

# --- 1e: Visualization ---
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Panel A: Argentina timeline
ax = axes[0]
yrs = arg['year'].values
if 'rmb_invoicing_w' in arg.columns:
    ax.plot(yrs, arg['rmb_invoicing_w'], 's-', color='#2980b9', lw=2, ms=6, label='RMB invoicing (%)')
if 'rmb_infra_it' in arg.columns:
    ax2 = ax.twinx()
    ax2.plot(yrs, arg['rmb_infra_it'], 'D--', color='#27ae60', lw=1.5, ms=5, label='RMB infra index')
    ax2.fill_between(yrs, 0, arg[SANC]*5, alpha=0.15, color='#c0392b', label='Sanctions (×5)')
    ax2.set_ylabel('Infra index / Sanctions×5')
    ax2.legend(fontsize=7, loc='upper left')
ax.axvline(2022, color='gray', ls='--', alpha=0.5)
ax.annotate('BOP crisis +\nswap line activation', xy=(2023, arg[arg['year']==2023]['rmb_invoicing_w'].iloc[0] if len(arg[arg['year']==2023])>0 else 0),
            xytext=(2020, arg['rmb_invoicing_w'].max()*0.8),
            fontsize=8, arrowprops=dict(arrowstyle='->', color='#c0392b'),
            bbox=dict(boxstyle='round,pad=0.3', facecolor='lightyellow'))
ax.set_xlabel('Year'); ax.set_ylabel('RMB Invoicing Share (%)')
ax.set_title('(a) Argentina: Push×Pull Activation Timeline', fontweight='bold')
ax.legend(fontsize=7, loc='lower left')

# Panel B: Leave-one-out coefficient distribution
ax = axes[1]
sorted_dfb = dfb_df.sort_values('beta_excl')
colors = ['#c0392b' if iso=='ARG' else '#bdc3c7' for iso in sorted_dfb['iso3']]
bars = ax.barh(range(len(sorted_dfb)), sorted_dfb['beta_excl'], color=colors, height=0.6)
ax.axvline(b_full, color='#2c3e50', ls='--', lw=2, label=f'Full sample: {b_full:+.1f}')
ax.axvline(0, color='black', lw=0.8)
# Label ARG bar
arg_idx = list(sorted_dfb['iso3']).index('ARG') if 'ARG' in sorted_dfb['iso3'].values else -1
if arg_idx >= 0:
    ax.text(sorted_dfb.iloc[arg_idx]['beta_excl']+0.5, arg_idx, 'ARG',
            fontsize=9, fontweight='bold', color='#c0392b', va='center')
# Label a few others
for i, (_, row) in enumerate(sorted_dfb.head(3).iterrows()):
    idx_pos = list(sorted_dfb['iso3']).index(row['iso3'])
    if row['iso3'] != 'ARG':
        ax.text(row['beta_excl']-1, idx_pos, row['iso3'], fontsize=7, va='center', ha='right')
for i, (_, row) in enumerate(sorted_dfb.tail(3).iterrows()):
    idx_pos = list(sorted_dfb['iso3']).index(row['iso3'])
    if row['iso3'] != 'ARG':
        ax.text(row['beta_excl']+0.5, idx_pos, row['iso3'], fontsize=7, va='center')

ax.set_yticks([])
ax.set_xlabel('Phase 2 Interaction Coefficient (excl. country)')
ax.set_title('(b) Leave-One-Out: Phase 2 Interaction Sensitivity', fontweight='bold')
ax.legend(fontsize=9)

fig.tight_layout()
fig.savefig(os.path.join(OUT, 'fig_argentina_deep_dive.png'), dpi=300, bbox_inches='tight')
plt.show()
print('  Saved: fig_argentina_deep_dive.png')

### 15.2 Marginal Effects Plot

In [ ]:
# ================================================================
# CELL 2: MARGINAL EFFECTS PLOT
# ================================================================
print('='*70)
print('MARGINAL EFFECTS: dRMB/dSanctions as f(Infrastructure)')
print('='*70)

# Fit full-sample interaction model
formula = ('rmb_invoicing_w ~ rmb_infra_it + '+SANC+' + infra_x_sanction + '
           +CS+' + EntityEffects + TimeEffects')
m = fit_twfe(df, formula)

b_sanc = m.params[SANC]
b_inter = m.params['infra_x_sanction']
se_sanc = m.std_errors[SANC]
se_inter = m.std_errors['infra_x_sanction']

# Covariance between sanctions and interaction coefficients
# From the full variance-covariance matrix
vcov = m.cov
idx_s = list(m.params.index).index(SANC)
idx_i = list(m.params.index).index('infra_x_sanction')
var_s = vcov.iloc[idx_s, idx_s]
var_i = vcov.iloc[idx_i, idx_i]
cov_si = vcov.iloc[idx_s, idx_i]

# Marginal effect: dY/dSanctions = b_sanc + b_inter * Infra
infra_grid = np.linspace(0, 2, 200)
me = b_sanc + b_inter * infra_grid
me_se = np.sqrt(var_s + infra_grid**2 * var_i + 2 * infra_grid * cov_si)
me_lo = me - 1.96 * me_se
me_hi = me + 1.96 * me_se

# Threshold: where marginal effect = 0
threshold = -b_sanc / b_inter
print(f'Sanctions coefficient: {b_sanc:.4f}')
print(f'Interaction coefficient: {b_inter:.4f}')
print(f'Sign-reversal threshold: Infra = {threshold:.3f}')
print(f'  When Infra < {threshold:.2f}: sanctions REINFORCE dollar (Phase 1)')
print(f'  When Infra > {threshold:.2f}: sanctions ERODE dollar (Phase 2)')

# Histogram of actual infra values
infra_actual = df['rmb_infra_it'].dropna()

fig, ax = plt.subplots(figsize=(10, 6))

# Shading: Phase 1 (red) and Phase 2 (green)
ax.fill_between(infra_grid, me_lo, me_hi, alpha=0.12, color='steelblue')
ax.fill_between(infra_grid[me<0], 0, me[me<0], alpha=0.08, color='#c0392b')
ax.fill_between(infra_grid[me>0], 0, me[me>0], alpha=0.08, color='#27ae60')

# Main line
ax.plot(infra_grid, me, '-', color='#2c3e50', lw=2.5, label='Marginal effect of sanctions')
ax.plot(infra_grid, me_lo, '--', color='steelblue', lw=0.8, alpha=0.6)
ax.plot(infra_grid, me_hi, '--', color='steelblue', lw=0.8, alpha=0.6)

# Zero line and threshold
ax.axhline(0, color='black', lw=1)
ax.axvline(threshold, color='#e67e22', ls=':', lw=2, alpha=0.8)

# Threshold annotation
ax.annotate(f'Sanctions Paradox\nthreshold: Infra = {threshold:.2f}',
            xy=(threshold, 0), xytext=(threshold+0.35, b_sanc*0.4),
            fontsize=10, fontweight='bold', color='#e67e22',
            arrowprops=dict(arrowstyle='->', color='#e67e22', lw=1.5),
            bbox=dict(boxstyle='round,pad=0.4', facecolor='lightyellow', alpha=0.9))

# Phase labels
ax.text(threshold*0.3, b_sanc*0.6, 'Phase 1:\nCoercive\nReinforcement',
        fontsize=9, color='#c0392b', ha='center', fontstyle='italic', alpha=0.8)
ax.text(min(threshold+0.6, 1.5), b_inter*0.3, 'Phase 2:\nCredibility\nErosion',
        fontsize=9, color='#27ae60', ha='center', fontstyle='italic', alpha=0.8)

# Rug plot of actual infrastructure values
ax2 = ax.twinx()
ax2.hist(infra_actual, bins=30, alpha=0.15, color='gray', density=True)
ax2.set_ylabel('Density of observations', color='gray', alpha=0.6)
ax2.tick_params(axis='y', colors='gray', labelsize=7)
ax2.set_ylim(0, ax2.get_ylim()[1]*3)  # shrink histogram

ax.set_xlabel('RMB Infrastructure Index (0–2)')
ax.set_ylabel('Marginal Effect of Sanctions on RMB Invoicing (pp)')
ax.set_title('Marginal Effect of Sanctions on RMB Invoicing\n'
             'as a Function of RMB Infrastructure (with 95% CI)',
             fontweight='bold')
ax.set_xlim(-0.05, 2.05)
ax.legend(loc='lower right', fontsize=9)

fig.tight_layout()
fig.savefig(os.path.join(OUT, 'fig_marginal_effects.png'), dpi=300, bbox_inches='tight')
plt.show()
print('  Saved: fig_marginal_effects.png')

### 15.3 Permutation Inference for Phase 2

In [ ]:
# ================================================================
# CELL 3: PERMUTATION INFERENCE FOR PHASE 2
# ================================================================
print('='*70)
print('PERMUTATION INFERENCE: Phase 2 Interaction')
print('='*70)

np.random.seed(42)
N_PERM = 1000

p2 = df[df['year']>=2022].copy()
formula = ('rmb_invoicing_w ~ rmb_infra_it + '+SANC+' + infra_x_sanction + '
           +CS+' + EntityEffects + TimeEffects')

# Actual coefficient
m_actual = fit_twfe(p2, formula)
b_actual = m_actual.params['infra_x_sanction']
print(f'Actual Phase 2 interaction: {b_actual:.3f}')

# Permutation: shuffle infra_x_sanction within each year
perm_coefs = []
for i in range(N_PERM):
    p2_perm = p2.copy()
    for yr in p2_perm['year'].unique():
        mask = p2_perm['year']==yr
        vals = p2_perm.loc[mask, 'infra_x_sanction'].values.copy()
        np.random.shuffle(vals)
        p2_perm.loc[mask, 'infra_x_sanction'] = vals
    try:
        m_p = fit_twfe(p2_perm, formula)
        perm_coefs.append(m_p.params['infra_x_sanction'])
    except:
        pass

perm_arr = np.array(perm_coefs)
perm_p = np.mean(np.abs(perm_arr) >= abs(b_actual))
print(f'Permutation distribution: mean={perm_arr.mean():.3f}, SD={perm_arr.std():.3f}')
print(f'Permutation p-value (two-sided): {perm_p:.4f}')
print(f'  (actual |{b_actual:.1f}| vs. permutation 95th: {np.percentile(np.abs(perm_arr), 95):.1f})')

# Plot
fig, ax = plt.subplots(figsize=(9, 5))
ax.hist(perm_arr, bins=50, alpha=0.6, color='#bdc3c7', edgecolor='white',
        label=f'Permutation distribution (N={len(perm_arr)})')
ax.axvline(b_actual, color='#c0392b', lw=2.5, ls='-',
           label=f'Actual: {b_actual:+.1f} (perm. p = {perm_p:.3f})')
ax.axvline(-b_actual, color='#c0392b', lw=1.5, ls='--', alpha=0.5)

# Shade rejection region
crit = np.percentile(np.abs(perm_arr), 95)
ax.axvspan(crit, perm_arr.max()*1.1, alpha=0.1, color='#c0392b')
ax.axvspan(-perm_arr.max()*1.1, -crit, alpha=0.1, color='#c0392b')

ax.set_xlabel('Interaction Coefficient (infra × sanctions)')
ax.set_ylabel('Frequency')
ax.set_title('Permutation Inference: Phase 2 Push×Pull Interaction\n'
             f'(1,000 permutations; two-sided p = {perm_p:.3f})',
             fontweight='bold')
ax.legend(fontsize=9)
fig.tight_layout()
fig.savefig(os.path.join(OUT, 'fig_permutation_test.png'), dpi=300, bbox_inches='tight')
plt.show()
print('  Saved: fig_permutation_test.png')

### 15.4 Placebo Treatment Date Test

In [ ]:
# ================================================================
# CELL 4: PLACEBO TREATMENT DATE TEST
# ================================================================
print('='*70)
print('PLACEBO: Fake CIPS accession 3 years earlier')
print('='*70)

df_placebo = df.copy()
# Shift treatment 3 years earlier
df_placebo['g_placebo'] = df_placebo['g_first_treat'] - 3
df_placebo['CIPS_placebo'] = np.where(
    df_placebo['g_placebo'].notna() & (df_placebo['year'] >= df_placebo['g_placebo']),
    1, 0)
df_placebo['treated_placebo'] = df_placebo['g_placebo'].notna().astype(int)
df_placebo['trend_placebo'] = df_placebo['treated_placebo']*(df_placebo['year']-df_placebo['year'].min())

# Restrict to pre-actual-treatment period to avoid contamination
df_pre = df_placebo[df_placebo['year'] < df_placebo['g_first_treat'].fillna(2099)].copy()

results_placebo = []
for dv, label in [('clearing_bank_it', 'Clearing bank'), ('rmb_invoicing_w', 'RMB invoicing')]:
    if dv not in df_pre.columns: continue
    try:
        # Actual
        m_a = fit_twfe(df, dv+' ~ CIPSit + trend_direct + '+CS+' + EntityEffects + TimeEffects')
        # Placebo
        m_p = fit_twfe(df_pre, dv+' ~ CIPS_placebo + trend_placebo + '+CS+' + EntityEffects + TimeEffects')

        results_placebo.append({
            'DV': label,
            'Actual_beta': round(m_a.params['CIPSit'], 4),
            'Actual_p': round(m_a.pvalues['CIPSit'], 4),
            'Placebo_beta': round(m_p.params['CIPS_placebo'], 4),
            'Placebo_p': round(m_p.pvalues['CIPS_placebo'], 4)
        })
        print(f'\n{label}:')
        print(f'  Actual:  beta = {m_a.params["CIPSit"]:+.4f} (p = {m_a.pvalues["CIPSit"]:.4f})')
        print(f'  Placebo: beta = {m_p.params["CIPS_placebo"]:+.4f} (p = {m_p.pvalues["CIPS_placebo"]:.4f})')
    except Exception as e:
        print(f'  {label}: {e}')

if results_placebo:
    pd.DataFrame(results_placebo).to_csv(os.path.join(OUT, 'table_placebo.csv'), index=False)
    print('\nInterpretation: if placebo is insignificant, the actual CIPS effect')
    print('is not an artifact of pre-existing trends.')
    print('  Saved: table_placebo.csv')

### 15.5 Integrated Force Composition Figure

In [ ]:
# ================================================================
# CELL 5: IMPROVED FORCE COMPOSITION FIGURE
# ================================================================
print('='*70)
print('FIGURE: Integrated Force Composition + Marginal Effects')
print('='*70)

# Two-row figure: top = marginal effects (main), bottom = vector diagrams
fig = plt.figure(figsize=(15, 11))
gs = gridspec.GridSpec(2, 2, height_ratios=[1.2, 1], hspace=0.3, wspace=0.25)

# ---- TOP: Marginal effects (full width) ----
ax_me = fig.add_subplot(gs[0, :])

# Re-use marginal effects computation
m_full_sample = fit_twfe(df, 'rmb_invoicing_w ~ rmb_infra_it + '+SANC+' + infra_x_sanction + '+CS+' + EntityEffects + TimeEffects')
b_s = m_full_sample.params[SANC]
b_i = m_full_sample.params['infra_x_sanction']
vcov = m_full_sample.cov
idx_s = list(m_full_sample.params.index).index(SANC)
idx_i = list(m_full_sample.params.index).index('infra_x_sanction')
var_s, var_i, cov_si = vcov.iloc[idx_s,idx_s], vcov.iloc[idx_i,idx_i], vcov.iloc[idx_s,idx_i]

ig = np.linspace(0, 2, 200)
me = b_s + b_i*ig
me_se = np.sqrt(var_s + ig**2*var_i + 2*ig*cov_si)
threshold = -b_s/b_i

ax_me.fill_between(ig, me-1.96*me_se, me+1.96*me_se, alpha=0.12, color='steelblue')
ax_me.fill_between(ig[me<0], 0, me[me<0], alpha=0.06, color='#c0392b')
ax_me.fill_between(ig[me>0], 0, me[me>0], alpha=0.06, color='#27ae60')
ax_me.plot(ig, me, '-', color='#2c3e50', lw=2.5)
ax_me.axhline(0, color='black', lw=1)
ax_me.axvline(threshold, color='#e67e22', ls=':', lw=2)

ax_me.text(0.3, -0.4, 'Phase 1:\nFrustrated demand', fontsize=8,
           color='#c0392b', alpha=0.6, ha='center', style='italic')
ax_me.text(1.5, 0.3, 'Phase 2:\nActivation zone', fontsize=8,
           color='#27ae60', alpha=0.6, ha='center', style='italic')


# Mark specific countries/regions on the marginal effects line
example_pts = [
    (0, 'No infra\n(Q1/Q3)', '#c0392b'),
    (1, 'Clearing OR\nswap (Q2)', '#8e44ad'),
    (2, 'Both\n(Q4)', '#27ae60'),
]
for infra_val, label, color in example_pts:
    me_val = b_s + b_i*infra_val
    ax_me.plot(infra_val, me_val, 'o', color=color, ms=10, zorder=5)
    ax_me.annotate(f'{label}\n({me_val:+.2f} pp)',
                   xy=(infra_val, me_val),
                   xytext=(infra_val+0.12, me_val + (0.4 if me_val<0 else -0.4)),
                   fontsize=8, color=color, fontweight='bold',
                   arrowprops=dict(arrowstyle='->', color=color, lw=1))

ax_me.annotate(f'Activation threshold\nInfra* = {threshold:.2f}\n(= |α₂|/α₃ = τᵢ condition)',
               xy=(threshold, 0), xytext=(threshold+0.3, b_s*0.5),
               fontsize=9, fontweight='bold', color='#e67e22',
               arrowprops=dict(arrowstyle='->', color='#e67e22', lw=1.5),
               bbox=dict(boxstyle='round,pad=0.3', facecolor='lightyellow', alpha=0.9))

ax_me.set_xlabel('RMB Infrastructure Index')
ax_me.set_ylabel('Marginal Effect of Sanctions\non RMB Invoicing (pp)')
ax_me.set_title('(a) Sanctions Paradox: Marginal Effect of Sanctions as f(Infrastructure)',
                fontweight='bold', fontsize=12)
ax_me.set_xlim(-0.05, 2.05)

ax_me.text(0.02, 0.95,
           f'From Equation (A):\n'
           f'  α₂ = {b_s:.2f} (sanctions main)\n'
           f'  α₃ = {b_i:+.2f} (interaction)\n'
           f'  Infra* = |α₂|/α₃ = {threshold:.2f}',
           transform=ax_me.transAxes, fontsize=9, va='top', family='serif',
           bbox=dict(boxstyle='round,pad=0.4', facecolor='white', alpha=0.8))


# ---- BOTTOM: Vector diagrams ----
P1_PUSH_Z, P2_PUSH_Z = 0.114, 1.840
P1_PULL_Z, P2_PULL_Z = 0.556, 1.046
P1_RESIST_Z, P2_RESIST_Z = 0.0, 1.5
P1_ANGLE, P2_ANGLE = np.radians(70), np.radians(35)

def make_vecs(push_mag, pull_mag, angle, resist_mag):
    midline = np.radians(45)
    push = push_mag*np.array([np.cos(midline+angle/2), np.sin(midline+angle/2)])
    pull = pull_mag*np.array([np.cos(midline-angle/2), np.sin(midline-angle/2)])
    net = push+pull
    resist = np.array([0.,0.])
    if resist_mag>0.01 and np.linalg.norm(net)>0.01:
        resist = -net/np.linalg.norm(net)*resist_mag*0.5
    return push, pull, net, resist, net+resist

for idx, (pz,plz,rz,ang,title,yr,inter) in enumerate([
    (P1_PUSH_Z, P1_PULL_Z, P1_RESIST_Z, P1_ANGLE,
     '(b) Phase 1 (Pre-2022)\nFrustrated demand → Marginal activation',
     'Pre-2022', '+1.58***'),
    (P2_PUSH_Z, P2_PULL_Z, P2_RESIST_Z, P2_ANGLE,
     '(c) Phase 2 (2022-23)\nActive diversification',
     '2022-2023', '+20.8**')]):

    ax = fig.add_subplot(gs[1, idx])
    sc = 1.3
    push,pull,net,resist,final = make_vecs(pz*sc,plz*sc,ang,rz*sc)
    o = np.array([0.,0.])
    ax.set_facecolor('#FAFAFA')
    pe_w = [pe.withStroke(linewidth=3, foreground='white')]

    for vec,col in [(push,'#c0392b'),(pull,'#27ae60')]:
        ax.annotate('',xy=vec,xytext=o,
                    arrowprops=dict(arrowstyle='-|>',color=col,lw=3.5,mutation_scale=18))
    ax.annotate('',xy=net,xytext=o,
                arrowprops=dict(arrowstyle='-|>',color='#2c3e50',lw=3,mutation_scale=16,ls='--'))
    if np.linalg.norm(resist)>0.05:
        ax.annotate('',xy=net+resist,xytext=net,
                    arrowprops=dict(arrowstyle='-|>',color='#e67e22',lw=3,mutation_scale=16))
    ax.annotate('',xy=final,xytext=o,
                arrowprops=dict(arrowstyle='-|>',color='#1a1a2e',lw=2.5,mutation_scale=14))

    ax.text(push[0]*0.3-0.15,push[1]*0.3+0.08,f'PUSH\n(Z={pz:.2f})',
            fontsize=8,fontweight='bold',color='#c0392b',ha='center',path_effects=pe_w)
    ax.text(pull[0]*0.7+0.05,pull[1]*0.7-0.15,f'PULL\n(Z={plz:.2f})',
            fontsize=8,fontweight='bold',color='#27ae60',ha='center',path_effects=pe_w)
    if np.linalg.norm(resist)>0.05:
        mr = net+resist*0.5
        ax.text(mr[0]+0.18,mr[1]+0.05,f'RESIST\n(Z={rz:.1f})',
                fontsize=8,fontweight='bold',color='#e67e22',ha='center',path_effects=pe_w)
    ax.text(final[0]+0.12,final[1]-0.08,'Net force',fontsize=7,
            color='#1a1a2e',fontstyle='italic',path_effects=pe_w)

    ax.text(0.95,0.05,f'Interaction: α₃ = {inter}',transform=ax.transAxes,fontsize=8,
            ha='right',va='bottom',
            bbox=dict(boxstyle='round,pad=0.3',facecolor='#E8F4FD',alpha=0.8))
    ax.text(0.05,0.95,yr,transform=ax.transAxes,fontsize=10,fontweight='bold',va='top',
            bbox=dict(boxstyle='round,pad=0.3',facecolor='#2c3e50',alpha=0.85),color='white')

    ax.set_xlabel('Infrastructure channel'); ax.set_ylabel('Political risk channel')
    ax.set_title(title, fontweight='bold')
    ax.grid(alpha=0.2,ls=':'); ax.set_aspect('equal')
    pts = np.array([o,push,pull,net,final,net+resist])
    pad=0.4
    ax.set_xlim(pts[:,0].min()-pad,pts[:,0].max()+pad)
    ax.set_ylim(pts[:,1].min()-pad,pts[:,1].max()+pad)

fig.suptitle('Force Composition of De-Dollarization',
             fontsize=14, fontweight='bold', y=0.98)
fig.savefig(os.path.join(OUT,'fig_force_composition_integrated.png'),
            dpi=300, bbox_inches='tight')
plt.show()
print('  Saved: fig_force_composition_integrated.png')

### 15.6 Supplementary Output Summary

In [ ]:
# ================================================================
# CELL 6: SUMMARY OF SUPPLEMENTARY RESULTS
# ================================================================
import glob
print('='*70)
print('v30 SUPPLEMENTARY OUTPUTS')
print('='*70)
outputs = sorted(glob.glob(os.path.join(OUT,'*')))
for f in outputs:
    print(f'  {os.path.basename(f):45s} {os.path.getsize(f):>8,d} bytes')

print(f'\nTotal: {len(outputs)} files')
print('\n--- INTEGRATION GUIDE ---')
print('1. fig_argentina_deep_dive.png → Section 8.5 (after leave-one-out discussion)')
print('   OR Appendix B as "Figure B1"')
print('2. fig_marginal_effects.png → Section 8.3 (REPLACE or accompany Table 2)')
print('   This is the MOST IMPACTFUL addition — standard in applied micro.')
print('   Consider making this Figure 2 in the main text.')
print('3. fig_permutation_test.png → Section 8.5 (supports Phase 2 credibility)')
print('   OR Appendix B as "Figure B2"')
print('4. table_placebo.csv → Section 8.7 Robustness (add as row in Table 6)')
print('5. fig_force_composition_integrated.png → Section 9.4 (REPLACE Figure 5)')
print('   Combines marginal effects + vector diagrams in one figure.')
print('\n--- THESIS TEXT TO ADD ---')
print('Section 8.3, after Table 2:')
print('  "Figure X presents the marginal effect of sanctions on RMB invoicing')
print('  as a continuous function of RMB infrastructure. The marginal effect')
print('  crosses zero at Infrastructure = [threshold], quantifying the Sanctions')
print('  Paradox threshold: below this level, sanctions reinforce dollar')
print('  dominance; above it, sanctions erode it. The 95% CI band excludes')
print('  zero for Infrastructure > [X], confirming statistical significance')
print('  of the sign reversal."')